# 1) Install dependencies

In [0]:
%pip install seaborn -q

# 2) Importing packages

In [0]:
# 2) Importing the needed packages
from pyspark.sql.functions import (
    col, count as _count, countDistinct, min as spark_min, max as spark_max,
    when, isnan, isnull, desc, lit, length, trim, lower, upper, current_date,
    sum as _sum, year, month, date_format, explode, split, regexp_extract,
    mean, stddev, abs as spark_abs, skewness, kurtosis, size, array,
    percentile_approx, struct, coalesce
)
from pyspark.sql.types import StringType, DoubleType, LongType, StructType
import re
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 3) Reading raw data

In [0]:
# 3.1) Reading raw data from Unity Catalog
TABLE_NAME = "yelp_poc_uc.bronze_final.yelp_tip"

In [0]:

# 3.2) Verify Delta format
table_info = spark.sql(f"DESCRIBE DETAIL {TABLE_NAME}").collect()[0]
assert table_info["format"].lower() == "delta", f"Expected Delta but found '{table_info['format']}'"
print(f"✅ Confirmed: {TABLE_NAME} is a Delta table")


In [0]:
# 3.3) Assign the table to a variable
df = spark.table(TABLE_NAME)

print(f"✅ Data loaded successfully from: {TABLE_NAME}")

# 4) General Exploration

In [0]:
# 4.1) Head - first 5 rows
print("\n📋 First 5 rows:")
display(df.limit(5))

In [0]:
# 4.2) Tail - last 6 rows (by business_id desc)
print("\n📋 Last 6 rows:")
display(df.orderBy(col("business_id").desc()).limit(6))


In [0]:
# 4.3) Dataset Info
total_rows = df.count()
print(f"\n📊 Dataset Info:")
print(f"   Total rows: {total_rows:,}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Top-level columns: {df.columns}")

In [0]:
# 4.4) Schema info
print("\n📋 Schema Info:")
df.printSchema()

In [0]:
# 4.5) Identify struct (nested) columns
struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
print(f"   Nested struct columns: {struct_cols}")

In [0]:
# 4.6) Missing values summary
print(f"\n📊 Missing Values Summary:")
for c in df.columns:
    null_count = df.filter(col(c).isNull()).count()
    if null_count > 0:
        print(f"   {c}: {null_count:,} nulls ({null_count/total_rows*100:.1f}%)")
    else:
        print(f"   {c}: ✅ No nulls")

In [0]:
# 4.7) Duplicate analysis
print(f"\n📊 Duplicate Analysis:")
total_rows_dup = df.count()
distinct_rows = df.distinct().count()
duplicate_rows = total_rows_dup - distinct_rows
print(f"   Total rows: {total_rows_dup:,}")
print(f"   Distinct rows: {distinct_rows:,}")
print(f"   Duplicate rows: {duplicate_rows:,} ({duplicate_rows/total_rows_dup*100:.2f}%)")

# Check duplicates by business_id (primary key candidate)
bid_total = df.select("business_id").count()
bid_distinct = df.select("business_id").distinct().count()
bid_duplicates = bid_total - bid_distinct
print(f"\n   Duplicates by 'business_id':")
print(f"   Total business_id: {bid_total:,}")
print(f"   Distinct business_id: {bid_distinct:,}")
print(f"   Duplicate business_id: {bid_duplicates:,} ({bid_duplicates/bid_total*100:.2f}%)")

if duplicate_rows == 0:
    print("\n   ✅ No duplicate rows found.")
else:
    print("\n   ⚠️ Duplicate rows detected! Consider deduplication.")

# 5) Structural & Meta-Data Exploration

In [0]:
# 5.1) Dimensionality/Shape
total_rows = df.count()
total_cols = len(df.columns)
print(f"Shape: {total_rows:,} rows × {total_cols:,} columns")   

In [0]:
# 5.2) Data Types and Data Classification

print("="*80)
print("RAW DATA TYPES")
print("="*80)
df.printSchema()

# Collect dtype info and schema-level nullable metadata
dtypes = dict(df.dtypes)
schema_nullable = {field.name: field.nullable for field in df.schema.fields}
total_rows = df.count()

for column in df.columns:
    dtype = dtypes[column]
    n = total_rows
    unique = df.select(countDistinct(col(column))).collect()[0][0]
    null_count = df.filter(col(column).isNull()).count()
    null_pct = null_count / n * 100 if n > 0 else 0
    unique_pct = unique / n * 100 if n > 0 else 0
    col_lower = column.lower()
    
    # Sample values (up to 3)
    samples = df.select(col(column)).filter(col(column).isNotNull()).distinct().limit(3)
    sample_values = [row[0] for row in samples.collect()]
    
    # ---- CLASSIFICATION ----
    if unique == 1:
        classification = "CONSTANT"
    elif col_lower in ['id', 'host_id']:
        classification = "IDENTIFIER"
    elif dtype in ('timestamp', 'date'):
        classification = "DATETIME"
    elif col_lower in ['latitude', 'longitude']:
        classification = "COORDINATE"
    elif dtype in ('int', 'bigint'):
        min_val = df.select(spark_min(col(column))).collect()[0][0]
        if min_val is not None and min_val >= 0:
            if unique == 2:
                classification = "BINARY_FLAG"
            elif unique <= 3:
                classification = "ORDINAL"
            elif unique <= 10:
                classification = "ORDINAL"
            else:
                classification = "COUNT"
        else:
            classification = "NUMERIC_INTEGER"
    elif dtype in ('double', 'float'):
        classification = "CONTINUOUS"
    elif dtype == 'string':
        if unique <= 5:
            classification = "CATEGORICAL"
        elif unique > total_rows * 0.5:
            classification = "FREE_TEXT"
        else:
            classification = "CATEGORICAL"
    else:
        classification = "UNCLASSIFIED"
    
    print(f"\n{'='*60}")
    print(f"Column: {column}")
    print(f"{'='*60}")
    nullable = schema_nullable.get(column, None)
    print(f"  Classification:      {classification}")
    print(f"  Data Type:           {dtype}")
    print(f"  Schema Nullable:     {nullable}")
    print(f"  Unique Values:       {unique:,} ({unique_pct:.1f}%)")
    print(f"  Missing Values:      {null_count} ({null_pct:.1f}%)")
    if nullable and null_count == 0:
        print(f"  ⚠️  Schema allows NULLs but none found — consider adding a NOT NULL constraint")
    elif not nullable and null_count > 0:
        print(f"  🚨  Schema disallows NULLs but {null_count} found — data integrity issue!")
    print(f"  Sample Values:       {', '.join(str(v) for v in sample_values)}")
    
    # ---- NUMERIC PROFILING ----
    if dtype in ('int', 'bigint', 'double', 'float'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        
        mean_val = df.select(mean(col(column))).collect()[0][0]
        median_val = df.select(percentile_approx(col(column), 0.5)).collect()[0][0]
        stddev_val = df.select(stddev(col(column))).collect()[0][0]
        skew_val = df.select(skewness(col(column))).collect()[0][0]
        
        zero_count = df.filter(col(column) == 0).count()
        neg_count = df.filter(col(column) < 0).count()
        
        q1 = df.select(percentile_approx(col(column), 0.25)).collect()[0][0]
        q3 = df.select(percentile_approx(col(column), 0.75)).collect()[0][0]
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outlier_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
        
        if skew_val is not None:
            if abs(skew_val) < 0.5:
                skew_text = "Symmetric"
            elif abs(skew_val) < 1:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (Moderate)"
            else:
                skew_text = f"{'Right' if skew_val > 0 else 'Left'}-skewed (High)"
        else:
            skew_text = "N/A"
        
        print(f"  Min:                 {min_val}")
        print(f"  Max:                 {max_val}")
        print(f"  Mean:                {mean_val:.2f}")
        print(f"  Median:              {median_val}")
        print(f"  Std Dev:             {stddev_val:.2f}")
        print(f"  Zero Count:          {zero_count}")
        print(f"  Negative Count:      {neg_count}")
        print(f"  Skewness:            {skew_val:.2f} ({skew_text})")
        print(f"  IQR Outliers:        {outlier_count} ({'Has outliers' if outlier_count > 0 else 'No outliers'})")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "COORDINATE":
            action = "Use for geospatial analysis"
        elif classification == "ORDINAL":
            action = "Preserve order (ordinal encoding)"
        elif classification == "BINARY_FLAG":
            action = "Use as boolean (0/1)"
        elif zero_count > total_rows * 0.9:
            action = "High zero percentage - consider log transform"
        elif skew_val and abs(skew_val) > 1:
            action = "Scale & check outliers (high skew)"
        else:
            action = "Scale & check outliers"
        print(f"  Action:              {action}")
    
    # ---- STRING PROFILING ----
    elif dtype == 'string':
        empty_count = df.filter((col(column) == "") | (col(column).isNull())).count()
        whitespace_count = df.filter(length(col(column)) != length(trim(col(column)))).count()
        
        total_non_null = n - null_count
        if total_non_null > 0:
            all_upper = df.filter(col(column) == upper(col(column))).count() - null_count
            all_lower = df.filter(col(column) == lower(col(column))).count() - null_count
            upper_pct = all_upper / total_non_null * 100
            lower_pct = all_lower / total_non_null * 100
            
            if upper_pct > 80:
                casing = f"{upper_pct:.0f}% Uppercase"
            elif lower_pct > 80:
                casing = f"{lower_pct:.0f}% Lowercase"
            else:
                casing = "Mixed Casing"
        else:
            casing = "N/A (all null)"
        
        len_stats = df.select(
            spark_min(length(col(column))).alias("min_len"),
            spark_max(length(col(column))).alias("max_len"),
            mean(length(col(column))).alias("avg_len")
        ).collect()[0]
        
        print(f"  Empty Count:         {empty_count}")
        print(f"  Whitespace Issues:   {whitespace_count}")
        print(f"  Casing:              {casing}")
        print(f"  Length:              Min: {len_stats['min_len']}, Max: {len_stats['max_len']}, Avg: {len_stats['avg_len']:.2f}")
        
        if classification == "IDENTIFIER":
            action = "Use as key only - exclude from modeling"
        elif classification == "FREE_TEXT":
            action = "Use for text mining / NLP only"
        elif unique <= 5:
            action = "One-hot or frequency encode"
        elif unique <= 50:
            action = "Label encode or one-hot (careful with cardinality)"
        else:
            action = "Frequency encode or target encode"
        print(f"  Action:              {action}")
    
    # ---- DATETIME PROFILING ----
    elif dtype in ('timestamp', 'date'):
        min_max = df.select(spark_min(col(column)), spark_max(col(column))).collect()[0]
        min_val = min_max[0]
        max_val = min_max[1]
        future_count = df.filter(col(column) > current_date()).count()
        old_count = df.filter(col(column) < "2008-01-01").count()
        
        print(f"  Date Range:          {min_val} to {max_val}")
        print(f"  Future Dates:        {future_count}")
        print(f"  Before 2008:         {old_count}")
        print(f"  Action:              Convert to datetime, extract features")

In [0]:
# 5.3) Column Header Audit
import re

print("="*80)
print("COLUMN HEADER AUDIT — Databricks Naming Convention Check")
print("="*80)

# Databricks / Delta Lake column naming rules:
# - Must start with a letter or underscore
# - Can contain letters, digits, underscores, hyphens, spaces (with column mapping)
# - Should avoid special characters
# - Best practice: lowercase snake_case, no leading/trailing spaces

columns = df.columns
total_cols = len(columns)
issues_found = 0

for c in columns:
    issues = []

    # Check: starts with letter or underscore
    if not re.match(r'^[a-zA-Z_]', c):
        issues.append("❌ Does not start with a letter or underscore")

    # Check: contains only valid characters (letters, digits, underscores)
    if not re.match(r'^[a-zA-Z_][a-zA-Z0-9_]*$', c):
        issues.append("⚠️  Contains characters beyond [a-zA-Z0-9_]")

    # Check: leading or trailing whitespace
    if c != c.strip():
        issues.append("❌ Has leading/trailing whitespace")

    # Check: contains spaces
    if ' ' in c:
        issues.append("⚠️  Contains spaces (use underscores instead)")

    # Check: contains hyphens
    if '-' in c:
        issues.append("⚠️  Contains hyphens (use underscores instead)")

    # Check: uppercase letters (best practice is lowercase snake_case)
    if c != c.lower():
        issues.append("⚠️  Contains uppercase letters (prefer lowercase snake_case)")

    # Check: consecutive underscores
    if '__' in c:
        issues.append("⚠️  Contains consecutive underscores")

    # Check: excessively long name
    if len(c) > 128:
        issues.append("⚠️  Column name exceeds 128 characters")

    # Check: reserved SQL keywords (common ones)
    reserved_keywords = {
        'select', 'from', 'where', 'table', 'column', 'index', 'group',
        'order', 'by', 'as', 'join', 'on', 'insert', 'update', 'delete',
        'create', 'drop', 'alter', 'into', 'values', 'set', 'null',
        'not', 'and', 'or', 'in', 'is', 'between', 'like', 'exists',
        'having', 'limit', 'offset', 'union', 'all', 'distinct', 'case',
        'when', 'then', 'else', 'end', 'cast', 'primary', 'key',
        'foreign', 'references', 'default', 'constraint', 'check',
        'unique', 'database', 'schema', 'view', 'replace', 'if',
        'true', 'false', 'date', 'time', 'timestamp', 'interval',
        'row', 'rows', 'window', 'partition', 'range', 'current',
        'user', 'role', 'grant', 'revoke'
    }
    if c.lower() in reserved_keywords:
        issues.append("⚠️  Column name is a SQL reserved keyword")

    # Report
    if issues:
        issues_found += 1
        print(f"\n  Column: '{c}'")
        for issue in issues:
            print(f"    {issue}")
        # Suggest corrected name
        suggested = re.sub(r'[^a-zA-Z0-9_]', '_', c)  # replace invalid chars
        suggested = re.sub(r'_+', '_', suggested)       # collapse multiple underscores
        suggested = suggested.strip('_').lower()         # lowercase, strip edge underscores
        if suggested != c:
            print(f"    💡 Suggested rename: '{c}' → '{suggested}'")
    else:
        print(f"\n  Column: '{c}' — ✅ Compliant")

print(f"\n{'='*80}")
print(f"SUMMARY: {total_cols} columns checked, {issues_found} with issues, {total_cols - issues_found} compliant")
if issues_found == 0:
    print("✅ All column names follow Databricks naming conventions!")
else:
    print(f"⚠️  {issues_found} column(s) have naming convention issues — review suggestions above.")
print("="*80)

In [0]:
# 5.4) Memory Footprint Estimation
print("="*80)
print("MEMORY FOOTPRINT ESTIMATION")
print("="*80)

total_rows = df.count()

# Estimate bytes per column based on data type
type_sizes = {
    'string': 50,      # average estimate
    'int': 4,
    'bigint': 8,
    'double': 8,
    'float': 4,
    'boolean': 1,
    'timestamp': 8,
    'date': 4,
}

total_estimated_bytes = 0
results = []

for field in df.schema.fields:
    col_name = field.name
    dtype = field.dataType.simpleString()
    
    if dtype == 'string':
        # Use actual average length for strings
        avg_len_row = df.select(mean(length(col(col_name)))).collect()[0][0]
        bytes_per_row = int(avg_len_row) if avg_len_row else 0
    else:
        bytes_per_row = type_sizes.get(dtype, 8)
    
    col_total = bytes_per_row * total_rows
    total_estimated_bytes += col_total
    results.append({
        "Column": col_name,
        "Type": dtype,
        "Est. Bytes/Row": bytes_per_row,
        "Est. Total (MB)": round(col_total / (1024**2), 2)
    })

# Add totals row
results.append({
    "Column": "TOTAL",
    "Type": "",
    "Est. Bytes/Row": "",
    "Est. Total (MB)": round(total_estimated_bytes / (1024**2), 2)
})

memory_df = pd.DataFrame(results)
display(memory_df)

print(f"\n  Total Estimated Size: {total_estimated_bytes / (1024**2):.2f} MB ({total_estimated_bytes / (1024**3):.2f} GB)")

# Warn if large
if total_estimated_bytes > 1 * 1024**3:
    print("  ⚠️  Estimated size > 1 GB — consider partitioning or filtering for local operations.")
else:
    print("  ✅ Dataset fits comfortably in memory.")
print("="*80)

In [0]:
# 5.5) Column Constraints & Key Candidates
print("="*80)
print("COLUMN CONSTRAINTS & KEY CANDIDATES")
print("="*80)

# 1) Table properties (Delta table properties, owner, etc.)
print("\n📋 Table Properties:")
try:
    props_df = spark.sql(f"SHOW TBLPROPERTIES {TABLE_NAME}")
    display(props_df)
except Exception as e:
    print(f"   Could not retrieve table properties: {e}")

# 2) NOT NULL constraints per column
print("\n📋 NOT NULL Constraints per Column:")
print(f"{'Column':<30} {'Data Type':<20} {'Nullable':<10}")
print("-" * 60)
for field in df.schema.fields:
    nullable_str = "YES" if field.nullable else "NO (🔒 enforced)"
    print(f"{field.name:<30} {field.dataType.simpleString():<20} {nullable_str:<10}")

# 3) Inferred uniqueness — identify candidate keys
print("\n📋 Candidate Key Analysis:")
total_rows = df.count()
for c in df.columns:
    distinct_count = df.select(countDistinct(col(c))).collect()[0][0]
    null_count = df.filter(col(c).isNull()).count()
    uniqueness_ratio = distinct_count / total_rows * 100 if total_rows > 0 else 0

    if distinct_count == total_rows and null_count == 0:
        status = "🔑 Candidate Primary Key (unique, no nulls)"
    elif distinct_count == total_rows:
        status = "🔑 Unique but has NULLs"
    elif uniqueness_ratio > 95:
        status = f"🔶 Near-unique ({uniqueness_ratio:.1f}%)"
    elif distinct_count <= 2:
        status = f"🔘 Low cardinality ({distinct_count} values)"
    else:
        status = f"📊 {distinct_count:,} distinct ({uniqueness_ratio:.1f}%)"

    print(f"   {c:<30} {status}")

print("\n" + "="*80)
print("META-DATA EXPLORATION COMPLETE")
print("="*80)

In [0]:
# # 6.1.1) Semantic Data Type Violations
# # Instead of comparing schema types against themselves (which is circular and always passes),
# # this infers the SEMANTIC type from actual data values and flags mismatches with the stored type.

# from pyspark.sql.types import (
#     IntegerType, LongType, StringType, DoubleType, FloatType,
#     DateType, TimestampType, BooleanType, ArrayType, MapType, StructType
# )
# from datetime import datetime as _dt

# print("=" * 80)
# print("6.1) SEMANTIC DATA TYPE VIOLATIONS")
# print("    Infers the likely intended type from actual values vs. stored schema type")
# print("=" * 80)

# total_rows = df.count()
# findings = []

# for field in df.schema.fields:
#     col_name = field.name
#     actual_type = field.dataType
#     actual_type_str = actual_type.simpleString()

#     non_null_df = df.select(col(col_name)).filter(col(col_name).isNotNull())
#     non_null_count = non_null_df.count()
#     sample_values = [row[0] for row in non_null_df.distinct().limit(10).collect()]

#     violations = []
#     inferred_type = actual_type_str  # default: type is correct

#     # ── Detect nested / complex types ──
#     if isinstance(actual_type, ArrayType):
#         print(f"\n{'─'*60}")
#         print(f"📌 Column: {col_name}")
#         print(f"   🔎 Nested: ArrayType — element: {actual_type.elementType}")
#     elif isinstance(actual_type, MapType):
#         print(f"\n{'─'*60}")
#         print(f"📌 Column: {col_name}")
#         print(f"   🔎 Nested: MapType — key: {actual_type.keyType}, value: {actual_type.valueType}")
#     elif isinstance(actual_type, StructType):
#         sub = [f"{sf.name}: {sf.dataType.simpleString()}" for sf in actual_type.fields]
#         print(f"\n{'─'*60}")
#         print(f"📌 Column: {col_name}")
#         print(f"   🔎 Nested: StructType — fields: {', '.join(sub)}")

#     # ══════════════════════════════════════════════════════════════
#     # STRING columns: probe what the values actually represent
#     # ══════════════════════════════════════════════════════════════
#     if isinstance(actual_type, StringType) and non_null_count > 0:

#         # 1) Comma-separated / delimited multi-value check (highest priority)
#         comma_list_count = non_null_df.filter(col(col_name).contains(", ")).count()
#         comma_ratio = comma_list_count / non_null_count

#         if comma_ratio > 0.10:  # >10 % rows have comma-separated values
#             # Probe whether individual elements look like timestamps
#             sample_str = str(sample_values[0]) if sample_values else ""
#             elements = [e.strip() for e in sample_str.split(",")][:5]
#             ts_hits = 0
#             for elem in elements:
#                 try:
#                     _dt.strptime(elem, "%Y-%m-%d %H:%M:%S")
#                     ts_hits += 1
#                 except Exception:
#                     pass

#             if ts_hits == len(elements) and elements:
#                 inferred_type = "array<timestamp>"
#                 violations.append(
#                     f"🔶 {comma_ratio*100:.1f}% of rows contain comma-separated TIMESTAMPS packed in a single string"
#                 )
#                 violations.append("   → Recommended: split(',') then cast elements to TimestampType, or explode into rows")
#             else:
#                 inferred_type = "array<string>"
#                 violations.append(
#                     f"🔶 {comma_ratio*100:.1f}% of rows contain comma-separated values packed in a single string"
#                 )
#                 violations.append("   → Recommended: split into ArrayType or explode into rows")
#         else:
#             # 2) JSON-embedded check
#             json_count = non_null_df.filter(
#                 (trim(col(col_name)).startswith("{") & trim(col(col_name)).endswith("}")) |
#                 (trim(col(col_name)).startswith("[") & trim(col(col_name)).endswith("]"))
#             ).count()
#             json_ratio = json_count / non_null_count

#             if json_ratio > 0.50:
#                 inferred_type = "struct / map (JSON)"
#                 violations.append(f"🔶 {json_ratio*100:.1f}% of values appear to be JSON")
#                 violations.append("   → Recommended: parse with from_json() / schema_of_json()")
#             else:
#                 # 3) Numeric check
#                 castable_numeric = non_null_df.filter(col(col_name).cast("double").isNotNull()).count()
#                 numeric_ratio = castable_numeric / non_null_count

#                 # 4) Timestamp check
#                 castable_ts = non_null_df.filter(col(col_name).cast("timestamp").isNotNull()).count()
#                 ts_ratio = castable_ts / non_null_count

#                 # 5) Boolean-pattern check
#                 bool_patterns = ['true','false','yes','no','0','1','y','n','t','f']
#                 bool_count = non_null_df.filter(lower(trim(col(col_name))).isin(bool_patterns)).count()
#                 bool_ratio = bool_count / non_null_count

#                 # Determine best semantic match (highest confidence first)
#                 if numeric_ratio > 0.95:
#                     # Guard: check if values are truly numeric or just hex/alphanumeric IDs
#                     # that Spark's cast("double") coerces to null-free results
#                     has_alpha = non_null_df.filter(col(col_name).rlike("[a-zA-Z]")).count()
#                     alpha_ratio = has_alpha / non_null_count
#                     if alpha_ratio > 0.05:
#                         pass  # Not truly numeric — likely IDs/hashes; skip numeric inference
#                     else:
#                         castable_long = non_null_df.filter(col(col_name).cast("long").isNotNull()).count()
#                         if castable_long / non_null_count > 0.95:
#                             inferred_type = "long / integer"
#                             violations.append(f"🔶 {numeric_ratio*100:.1f}% castable to numeric (integer-like)")
#                             violations.append("   → Recommended: cast to LongType or IntegerType")
#                         else:
#                             inferred_type = "double / float"
#                             violations.append(f"🔶 {numeric_ratio*100:.1f}% castable to numeric (decimal)")
#                             violations.append("   → Recommended: cast to DoubleType")
#                 elif ts_ratio > 0.95:
#                     inferred_type = "timestamp"
#                     violations.append(f"🔶 {ts_ratio*100:.1f}% parseable as timestamp")
#                     violations.append("   → Recommended: cast to TimestampType")
#                 elif bool_ratio > 0.95:
#                     inferred_type = "boolean"
#                     violations.append(f"🔶 {bool_ratio*100:.1f}% match boolean patterns")
#                     violations.append("   → Recommended: cast to BooleanType")
#                 elif 0.05 < numeric_ratio <= 0.95:
#                     violations.append(f"⚠️  Mixed content: {numeric_ratio*100:.1f}% castable to numeric")
#                     bad_examples = (
#                         non_null_df.filter(col(col_name).cast("double").isNull())
#                         .distinct().limit(5)
#                     )
#                     violations.append(f"      Non-numeric examples: {[r[0] for r in bad_examples.collect()]}")

#     # ══════════════════════════════════════════════════════════════
#     # INTEGER / LONG columns: detect boolean-like or constant flags
#     # ══════════════════════════════════════════════════════════════
#     elif isinstance(actual_type, (IntegerType, LongType)) and non_null_count > 0:
#         min_val = df.agg(spark_min(col(col_name))).collect()[0][0]
#         max_val = df.agg(spark_max(col(col_name))).collect()[0][0]
#         distinct_count = df.select(countDistinct(col(col_name))).collect()[0][0]

#         if distinct_count == 2 and min_val == 0 and max_val == 1:
#             inferred_type = "boolean"
#             violations.append("ℹ️  Binary (0/1) — could be stored as BooleanType")
#         elif distinct_count == 1:
#             inferred_type = f"{actual_type_str} (constant)"
#             violations.append(f"ℹ️  Column is constant (single value: {min_val})")

#     # ══════════════════════════════════════════════════════════════
#     # DOUBLE / FLOAT columns: detect integer-only values
#     # ══════════════════════════════════════════════════════════════
#     elif isinstance(actual_type, (DoubleType, FloatType)) and non_null_count > 0:
#         frac_check = non_null_df.filter(
#             (col(col_name) != col(col_name).cast("long").cast(actual_type_str))
#         ).count()
#         if frac_check == 0:
#             inferred_type = "long / integer (no fractional part)"
#             violations.append("ℹ️  All values are whole numbers — consider LongType for efficiency")

#     # ── Report per column ──
#     status = "✅ PASS" if not violations else "⚠️  FINDING"
#     print(f"\n{'─'*60}")
#     print(f"📌 Column: {col_name}")
#     print(f"   Schema Type:    {actual_type_str}")
#     print(f"   Inferred Type:  {inferred_type}")
#     print(f"   Non-Null Count: {non_null_count:,} / {total_rows:,}")
#     print(f"   Status:         {status}")

#     if sample_values:
#         display_samples = [str(s)[:80] + ("..." if len(str(s)) > 80 else "") for s in sample_values[:3]]
#         print(f"   Samples:        {display_samples}")
#     for v in violations:
#         print(f"   {v}")

#     findings.append({
#         "Column": col_name,
#         "Schema Type": actual_type_str,
#         "Inferred Semantic Type": inferred_type,
#         "Status": "PASS" if not violations else "FINDING",
#         "Detail": "; ".join(v.strip() for v in violations) if violations else "Types aligned"
#     })

# # ── Summary ──
# print(f"\n{'='*80}")
# print("SUMMARY")
# print("="*80)
# findings_df = pd.DataFrame(findings)
# display(findings_df)

# violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
# print(f"\n{len(findings)} columns checked, {violation_count} finding(s)")
# if violation_count == 0:
#     print("✅ All column schema types match their inferred semantic types.")
# else:
#     print(f"⚠️  {violation_count} column(s) have schema-vs-semantic type mismatches — review above.")
# print("SEMANTIC DATA TYPE VIOLATIONS CHECK COMPLETE")
# print("=" * 80)

### Temporary Transformation 1 & 2

#### For this Comprehensive Assessment to be logical we will need to
1) Fix the Data Types
2) Remodel the data

#### Important Note: These fixes are won't change anything yet in the original tables, they are just made temporarily here till the cleaning step of the pipeline

In [0]:
# # Fix Data Types based on 6.1) Semantic Data Type Violations findings
# # ─────────────────────────────────────────────────────────────────────
# # business_id : string ✅  (alphanumeric hash IDs — correctly typed)
# # date        : string ⚠️   → split comma-separated timestamps into array<timestamp>
# #
# # Important: These fixes are TEMPORARY for this EDA notebook.
# # Original tables in bronze_final are NOT modified.

# print("=" * 80)
# print("FIX DATA TYPES")
# print("=" * 80)

# print("\n📋 BEFORE fix:")
# df.printSchema()

# # Split the comma-separated timestamp string into a proper array<timestamp>
# df_typed = (
#     df
#     .withColumn(
#         "checkin_dates",
#         split(col("date"), ",\\s*")          # split on comma + optional whitespace
#             .cast("array<timestamp>")         # cast each element to TimestampType
#     )
#     .drop("date")                              # drop original packed string column
# )

# print("\n📋 AFTER fix:")
# df_typed.printSchema()

# # Quick validation: check for null arrays (failed casts)
# null_arrays = df_typed.filter(col("checkin_dates").isNull()).count()
# total = df_typed.count()
# print(f"\n✅ Rows: {total:,}")
# print(f"   Null arrays (failed casts): {null_arrays:,}")

# # Spot-check: show a few rows with their typed arrays
# print("\n📋 Sample (typed):")
# display(df_typed.limit(5))

In [0]:
# # Remodel: Explode array so each checkin event becomes its own row
# # + extract temporal features for downstream analysis
# # ─────────────────────────────────────────────────────────────────────
# from pyspark.sql.functions import (
#     hour, minute, second, dayofweek, dayofmonth, dayofyear,
#     weekofyear, month, year, quarter, date_format, last_day,
#     next_day, trunc, months_between, lit, when, floor, ceil, concat
# )

# print("=" * 80)
# print("REMODEL DATA  —  Full Date-Dimension Extraction")
# print("=" * 80)

# df_remodeled = (
#     df_typed
#     # ── Explode: 1 row per business × N checkins → N rows per business × 1 checkin ──
#     .withColumn("checkin_timestamp_ISO_8601", explode(col("checkin_dates")))
#     .drop("checkin_dates")

#     # ══════════════════════════════════════════════════════════════════════
#     # DATE-LEVEL ATTRIBUTES  (date dimension style)
#     # ══════════════════════════════════════════════════════════════════════
#     .withColumn("checkin_date",        col("checkin_timestamp_ISO_8601").cast("date"))
#     .withColumn("checkin_year",        year(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_month",       month(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_month_name",  date_format(col("checkin_timestamp_ISO_8601"), "MMMM"))        # January…
#     .withColumn("checkin_month_abbr",  date_format(col("checkin_timestamp_ISO_8601"), "MMM"))          # Jan…
#     .withColumn("checkin_day",         dayofmonth(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_day_of_year", dayofyear(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_dow_number",  dayofweek(col("checkin_timestamp_ISO_8601")))                   # 1=Sun…7=Sat
#     .withColumn("checkin_dow_name",    date_format(col("checkin_timestamp_ISO_8601"), "EEEE"))         # Monday…
#     .withColumn("checkin_dow_abbr",    date_format(col("checkin_timestamp_ISO_8601"), "EEE"))          # Mon…
#     .withColumn("checkin_quarter",     quarter(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_quarter_name",concat(lit("Q"), quarter(col("checkin_timestamp_ISO_8601"))))
#     .withColumn("checkin_week_of_year",weekofyear(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_year_month",  date_format(col("checkin_timestamp_ISO_8601"), "yyyy-MM"))      # 2019-07
#     .withColumn("checkin_year_quarter",
#         concat(year(col("checkin_timestamp_ISO_8601")).cast("string"), lit("-Q"), quarter(col("checkin_timestamp_ISO_8601")).cast("string"))
#     )

#     # ── Half-year / Semester ──
#     .withColumn("checkin_half_year",
#         when(month(col("checkin_timestamp_ISO_8601")) <= 6, lit(1)).otherwise(lit(2))
#     )
#     .withColumn("checkin_half_year_name",
#         when(month(col("checkin_timestamp_ISO_8601")) <= 6, lit("H1")).otherwise(lit("H2"))
#     )

#     # ── Month position flags ──
#     .withColumn("checkin_month_start_date", trunc(col("checkin_timestamp_ISO_8601"), "month"))
#     .withColumn("checkin_month_end_date",   last_day(col("checkin_timestamp_ISO_8601")))
#     .withColumn("is_month_start",
#         (dayofmonth(col("checkin_timestamp_ISO_8601")) == 1).cast("boolean")
#     )
#     .withColumn("is_month_end",
#         (col("checkin_date") == last_day(col("checkin_timestamp_ISO_8601"))).cast("boolean")
#     )

#     # ── Year position flags ──
#     .withColumn("checkin_year_start_date", trunc(col("checkin_timestamp_ISO_8601"), "year"))
#     .withColumn("is_year_start",
#         ((month(col("checkin_timestamp_ISO_8601")) == 1) & (dayofmonth(col("checkin_timestamp_ISO_8601")) == 1)).cast("boolean")
#     )
#     .withColumn("is_year_end",
#         ((month(col("checkin_timestamp_ISO_8601")) == 12) & (dayofmonth(col("checkin_timestamp_ISO_8601")) == 31)).cast("boolean")
#     )

#     # ── Weekend / Weekday flag ──
#     .withColumn("is_weekend",
#         dayofweek(col("checkin_timestamp_ISO_8601")).isin(1, 7).cast("boolean")  # 1=Sun, 7=Sat
#     )
#     .withColumn("is_weekday",
#         (~dayofweek(col("checkin_timestamp_ISO_8601")).isin(1, 7)).cast("boolean")
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # TIME-LEVEL ATTRIBUTES
#     # ══════════════════════════════════════════════════════════════════════
#     .withColumn("checkin_hour",   hour(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_minute", minute(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_second", second(col("checkin_timestamp_ISO_8601")))
#     .withColumn("checkin_time_of_day",
#         when(hour(col("checkin_timestamp_ISO_8601")).between(5, 11),  lit("Morning"))
#         .when(hour(col("checkin_timestamp_ISO_8601")).between(12, 16), lit("Afternoon"))
#         .when(hour(col("checkin_timestamp_ISO_8601")).between(17, 20), lit("Evening"))
#         .otherwise(lit("Night"))
#     )
#     .withColumn("checkin_am_pm",
#         when(hour(col("checkin_timestamp_ISO_8601")) < 12, lit("AM")).otherwise(lit("PM"))
#     )
#     .withColumn("checkin_hour_12",
#         when(hour(col("checkin_timestamp_ISO_8601")) == 0, lit(12))
#         .when(hour(col("checkin_timestamp_ISO_8601")) > 12, hour(col("checkin_timestamp_ISO_8601")) - 12)
#         .otherwise(hour(col("checkin_timestamp_ISO_8601")))
#     )
#     .withColumn("checkin_hour_bin_4h",
#         (floor(hour(col("checkin_timestamp_ISO_8601")) / 4) * 4).cast("int")
#     )
# )

# # Reassign df and total_rows so ALL downstream cells use the remodeled data
# df = df_remodeled
# total_rows = df.count()

# print(f"\n📊 Remodeled shape: {total_rows:,} rows × {len(df.columns)} columns")
# print(f"   (Original: 131,930 rows × 2 cols → one row per individual checkin event)")
# print("\n📋 Schema:")
# df.printSchema()
# print("📋 Sample (remodeled):")
# display(df.limit(10))

In [0]:
# # 6.1.2) Invalid Categories
# # Values not found in a closed/allowed set (e.g., month names, day names, AM/PM)
# # ─────────────────────────────────────────────────────────────────────
# # After remodeling, several temporal columns have fixed categorical domains.
# # This check validates that every value belongs to the expected set.

# print("=" * 80)
# print("6.1.2) INVALID CATEGORIES")
# print("    Checking values against closed/allowed sets")
# print("=" * 80)

# total_rows = df.count()
# df_columns = set(df.columns)   # cache once (avoids repeated Analyze RPCs on Spark Connect)
# findings = []

# # ══ Define allowed value sets for each categorical column ══
# valid_months_full  = ["January","February","March","April","May","June",
#                       "July","August","September","October","November","December"]
# valid_months_abbr  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
# valid_dow_full     = ["Sunday","Monday","Tuesday","Wednesday","Thursday","Friday","Saturday"]
# valid_dow_abbr     = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]
# valid_time_of_day  = ["Morning","Afternoon","Evening","Night"]
# valid_am_pm        = ["AM","PM"]
# valid_quarter_name = ["Q1","Q2","Q3","Q4"]
# valid_half_year_name = ["H1","H2"]

# categorical_checks = [
#     ("checkin_month_name",    valid_months_full,    "Full month name"),
#     ("checkin_month_abbr",    valid_months_abbr,    "Abbreviated month name"),
#     ("checkin_dow_name",      valid_dow_full,       "Full day-of-week name"),
#     ("checkin_dow_abbr",      valid_dow_abbr,       "Abbreviated day-of-week name"),
#     ("checkin_time_of_day",   valid_time_of_day,    "Time of day bucket"),
#     ("checkin_am_pm",         valid_am_pm,          "AM/PM indicator"),
#     ("checkin_quarter_name",  valid_quarter_name,   "Quarter name"),
#     ("checkin_half_year_name",valid_half_year_name,  "Half year name"),
# ]

# for col_name, allowed, desc in categorical_checks:
#     if col_name in df_columns:
#         bad_count = df.filter(
#             ~col(col_name).isin(allowed) & col(col_name).isNotNull()
#         ).count()
#         status = "PASS" if bad_count == 0 else "FINDING"
#         icon = "✅" if bad_count == 0 else "⚠️ "
#         print(f"\n   {icon} {col_name}: {bad_count:,} violations — {status}")
#         print(f"      Allowed: {allowed}")
#         if bad_count > 0:
#             bad_vals = (
#                 df.filter(~col(col_name).isin(allowed) & col(col_name).isNotNull())
#                   .select(col_name).distinct().limit(10)
#             )
#             print(f"      ⚠️  Unexpected values: {[r[0] for r in bad_vals.collect()]}")
#         findings.append({
#             "Column": col_name,
#             "Allowed Values": f"{allowed[:3]}... ({len(allowed)} total)",
#             "Description": desc,
#             "Violations": bad_count,
#             "Status": status
#         })

# # ══ Summary ══
# print(f"\n{'='*80}")
# print("SUMMARY")
# print("=" * 80)
# findings_df = pd.DataFrame(findings)
# display(findings_df)

# violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
# print(f"\n{len(findings)} category checks performed, {violation_count} finding(s)")
# if violation_count == 0:
#     print("✅ All categorical columns contain only valid/allowed values.")
# else:
#     print(f"⚠️  {violation_count} column(s) have invalid categories — review above.")
# print("INVALID CATEGORIES CHECK COMPLETE")
# print("=" * 80)

In [0]:
# # 6.1.3) Pattern Mismatches
# # Format violations like date formats, email patterns, phone numbers, codes, etc.
# # ─────────────────────────────────────────────────────────────────────────────────
# # After remodeling, the checkin dataset contains:
# #   - business_id                : 22-char alphanumeric hash
# #   - checkin_timestamp_ISO_8601 : timestamp (cast from ISO 8601 string)
# #   - checkin_date               : date derived from timestamp
# #   - Temporal features          : derived numerics/strings from timestamp
# #
# # Checkable patterns:
# #   1) business_id should be exactly 22 alphanumeric characters (Yelp hash format)
# #   2) checkin_timestamp should conform to valid datetime ranges (no malformed casts)
# #   3) Derived temporal columns should be consistent with their source timestamp

# from pyspark.sql.functions import length, col, when, lit, sum as spark_sum

# print("=" * 80)
# print("6.1.3) PATTERN MISMATCHES")
# print("    Checking format violations across all columns")
# print("=" * 80)

# total_rows = df.count()
# findings = []

# # ══════════════════════════════════════════════════════════════════════════════
# # 1) business_id — Expected: 22-character alphanumeric hash (Yelp format)
# # ══════════════════════════════════════════════════════════════════════════════
# print(f"\n{'─'*60}")
# print("📌 Check 1: business_id — 22-char alphanumeric hash pattern")
# print("   Expected pattern: ^[a-zA-Z0-9_-]{22}$")

# # IDs that don't match the expected 22-char alphanumeric pattern
# bad_id_df = df.filter(
#     ~col("business_id").rlike("^[a-zA-Z0-9_-]{22}$")
# )
# bad_id_count = bad_id_df.count()

# if bad_id_count == 0:
#     print(f"   ✅ PASS — All {total_rows:,} rows have valid 22-char alphanumeric business_id")
# else:
#     print(f"   ⚠️  FINDING — {bad_id_count:,} / {total_rows:,} rows ({bad_id_count/total_rows*100:.2f}%) have non-conforming business_id")
#     bad_samples = [row[0] for row in bad_id_df.select("business_id").distinct().limit(10).collect()]
#     print(f"   Examples: {bad_samples}")

# findings.append({
#     "Column": "business_id",
#     "Pattern": "^[a-zA-Z0-9_-]{22}$",
#     "Description": "22-char alphanumeric Yelp hash",
#     "Violations": bad_id_count,
#     "Status": "PASS" if bad_id_count == 0 else "FINDING"
# })

# # ══════════════════════════════════════════════════════════════════════════════
# # 2) checkin_timestamp_ISO_8601 — Should be non-null valid timestamp
# #    (null = failed cast from original string → pattern mismatch)
# # ══════════════════════════════════════════════════════════════════════════════
# print(f"\n{'─'*60}")
# print("📌 Check 2: checkin_timestamp_ISO_8601 — Valid timestamp (non-null after cast)")

# null_ts_count = df.filter(col("checkin_timestamp_ISO_8601").isNull()).count()

# if null_ts_count == 0:
#     print(f"   ✅ PASS — All {total_rows:,} timestamps are valid (no null from failed casts)")
# else:
#     print(f"   ⚠️  FINDING — {null_ts_count:,} / {total_rows:,} rows ({null_ts_count/total_rows*100:.2f}%) have NULL timestamps (malformed source strings)")

# findings.append({
#     "Column": "checkin_timestamp_ISO_8601",
#     "Pattern": "Valid ISO 8601 timestamp",
#     "Description": "Non-null after cast from comma-separated string",
#     "Violations": null_ts_count,
#     "Status": "PASS" if null_ts_count == 0 else "FINDING"
# })

# # ══════════════════════════════════════════════════════════════════════════════
# # 3) checkin_date — Should match the date portion of the timestamp
# # ══════════════════════════════════════════════════════════════════════════════
# print(f"\n{'─'*60}")
# print("📌 Check 3: checkin_date — Consistent with checkin_timestamp_ISO_8601")

# date_mismatch_count = df.filter(
#     col("checkin_date") != col("checkin_timestamp_ISO_8601").cast("date")
# ).count()

# if date_mismatch_count == 0:
#     print(f"   ✅ PASS — All {total_rows:,} checkin_date values match their source timestamp")
# else:
#     print(f"   ⚠️  FINDING — {date_mismatch_count:,} / {total_rows:,} rows have date ≠ timestamp.cast('date')")

# findings.append({
#     "Column": "checkin_date",
#     "Pattern": "== checkin_timestamp.cast('date')",
#     "Description": "Date portion must match source timestamp",
#     "Violations": date_mismatch_count,
#     "Status": "PASS" if date_mismatch_count == 0 else "FINDING"
# })

# # ══════════════════════════════════════════════════════════════════════════════
# # SUMMARY
# # ══════════════════════════════════════════════════════════════════════════════
# print(f"\n{'='*80}")
# print("SUMMARY")
# print("=" * 80)

# findings_df = pd.DataFrame(findings)
# display(findings_df)

# violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
# print(f"\n{len(findings)} pattern checks performed, {violation_count} finding(s)")
# if violation_count == 0:
#     print("✅ All values conform to their expected patterns and formats.")
# else:
#     print(f"⚠️  {violation_count} check(s) detected pattern mismatches — review above.")
# print("PATTERN MISMATCHES CHECK COMPLETE")
# print("=" * 80)

In [0]:
# # 6.1.4) Range Violations
# # Values outside allowed min/max bounds for numeric/temporal columns
# # ─────────────────────────────────────────────────────────────────────

# print("=" * 80)
# print("6.1.4) RANGE VIOLATIONS")
# print("    Checking numeric/temporal values fall within physically valid bounds")
# print("=" * 80)

# total_rows = df.count()
# df_columns = set(df.columns)   # cache once (avoids repeated Analyze RPCs on Spark Connect)
# findings = []

# # ══ Define valid ranges: (column, min, max, description) ══
# range_checks = [
#     ("checkin_year",         1950, 2026, "Year"),
#     ("checkin_month",        1, 12,   "Month"),
#     ("checkin_day",          1, 31,   "Day of month"),
#     ("checkin_day_of_year",  1, 366,  "Day of year"),
#     ("checkin_dow_number",   1, 7,    "Day of week (1=Sun..7=Sat)"),
#     ("checkin_quarter",      1, 4,    "Quarter"),
#     ("checkin_week_of_year", 1, 53,   "Week of year"),
#     ("checkin_half_year",    1, 2,    "Half year"),
#     ("checkin_hour",         0, 23,   "Hour"),
#     ("checkin_minute",       0, 59,   "Minute"),
#     ("checkin_second",       0, 59,   "Second"),
#     ("checkin_hour_12",      1, 12,   "Hour (12h format)"),
#     ("checkin_hour_bin_4h",  0, 20,   "Hour bin (4h buckets: 0,4,8,12,16,20)"),
# ]

# for col_name, min_val, max_val, desc in range_checks:
#     if col_name in df_columns:
#         out_of_range = df.filter(
#             (col(col_name) < min_val) | (col(col_name) > max_val)
#         ).count()
#         status = "PASS" if out_of_range == 0 else "FINDING"
#         icon = "✅" if out_of_range == 0 else "⚠️ "
#         print(f"   {icon} {col_name} [{min_val}–{max_val}]: {out_of_range:,} violations — {status}")

#         if out_of_range > 0:
#             bad_vals = (
#                 df.filter((col(col_name) < min_val) | (col(col_name) > max_val))
#                   .select(col_name).distinct().limit(10)
#             )
#             print(f"      Out-of-range values: {[r[0] for r in bad_vals.collect()]}")

#         findings.append({
#             "Column": col_name,
#             "Valid Range": f"[{min_val}–{max_val}]",
#             "Description": f"{desc} within valid range",
#             "Violations": out_of_range,
#             "Status": status
#         })

# # ══ Timestamp plausibility (Yelp founded 2004, data should be ≤ today) ══
# print(f"\n{'─'*60}")
# print("📌 Timestamp plausibility (Yelp founded 2004, data should be ≤ today)")

# if "checkin_timestamp_ISO_8601" in df_columns:
#     from pyspark.sql.functions import min as spark_min, max as spark_max

#     ts_stats = df.agg(
#         spark_min("checkin_timestamp_ISO_8601").alias("min_ts"),
#         spark_max("checkin_timestamp_ISO_8601").alias("max_ts")
#     ).collect()[0]
#     print(f"   Earliest checkin: {ts_stats['min_ts']}")
#     print(f"   Latest checkin:   {ts_stats['max_ts']}")

#     implausible = df.filter(
#         (year(col("checkin_timestamp_ISO_8601")) < 2004) |
#         (col("checkin_timestamp_ISO_8601") > current_date())
#     ).count()

#     status = "PASS" if implausible == 0 else "FINDING"
#     icon = "✅" if implausible == 0 else "⚠️ "
#     print(f"   {icon} Implausible timestamps (before 2004 or future): {implausible:,} — {status}")

#     findings.append({
#         "Column": "checkin_timestamp_ISO_8601",
#         "Valid Range": "[2004 – today]",
#         "Description": "Timestamp within Yelp's operational period",
#         "Violations": implausible,
#         "Status": status
#     })

# # ══ Summary ══
# print(f"\n{'='*80}")
# print("SUMMARY")
# print("=" * 80)
# findings_df = pd.DataFrame(findings)
# display(findings_df)

# violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
# print(f"\n{len(findings)} range checks performed, {violation_count} finding(s)")
# if violation_count == 0:
#     print("✅ All numeric/temporal values fall within valid bounds.")
# else:
#     print(f"⚠️  {violation_count} column(s) have range violations — review above.")
# print("RANGE VIOLATIONS CHECK COMPLETE")
# print("=" * 80)

# 6) Data Quality Checks/Assessment


### 6.1) STRUCTURAL / SCHEMA CHECKS
*Verify the dataset's shape, schema, and declared types match expectations — and that declared types actually reflect the semantic content of the data.*

- **Column Presence & Order** — Expected columns exist; no missing or unexpected columns; order matches schema contract(if it exists).
  - *Example:* Expected `[id, fname, lname, hire_date]`, found `[id, name, hire_date]` → missing `fname`, `lname`; unexpected `name`.
  - *Columns present but reordered → flag if downstream consumers depend on positional access.*

- **Data Type Conformance** — Checks the column's declared type matches the type specified in the schema contract(if it exists).
  - *Examples:*
    - Contract says age INT, dataset declares age VARCHAR → schema mismatch.
    - Contract says hire_date DATE, dataset declares hire_date TIMESTAMP → widening mismatch (may be acceptable; configurable).
    - Contract says salary DECIMAL(10,2), dataset declares salary FLOAT → precision mismatch.

- **Semantic Data Type Violations** - The column's actual data values match the type the schema claims they are. The semantic type is inferred from the values themselves — not read from the schema — so the check is non-circular.
    - *Example:* A column can declare INT in the schema but contain "twenty-five", "unknown", "$1,200", or empty strings. Schema-vs-schema checks never catch this. Semantic inference does.

- **Schema Drift Detection** — Compare current schema against baseline/columns in the schema contract(if it exists); flag added/removed/retyped columns.
  - *Example:* `salary` changed from `INT` to `VARCHAR` between loads.

---

### 6.2) DOMAIN & RANGE CHECKS
*Verify values belong to allowed sets and fall within permitted bounds. All scope variants live here.*

- **Allowed Value Conformance (single column)** — Values belong to the permitted domain/category list.
  - *Example:* `sex` allows only `M`, `F`, `Other`; found `Male`, `1`, `Unknown`.
  - *Sub:* Geographic codes — `country_code` must be ISO-3166 alpha-2.
  - *Sub:* Department names — `department` must match organizational list.
  - *Sub:* Status codes — `employee_status` must be in `{Active, Terminated, Leave}`.

- **Range Conformance (single column)** — Values within min/max bounds.
  - *Example:* `age` must be 0–120; found `-3` or `250`.
  - *Sub:* Salary bands — `salary` within role-specific band.
  - *Sub:* Date bounds — `hire_date` after 1980-01-01.

- **Pattern Conformance** — Values match required format (email, phone, postal code, ID patterns).
  - *Example:* `email` must match RFC-5322; `phone` must match E.164.

- **Impossible Value Detection** — Values that cannot exist regardless of bounds.
  - *Example:* `birth_date` in the future; `age` negative; `death_date` before `birth_date` (single-column view).

---

### 6.3) FORMAT & ENCODING STANDARDIZATION
*Detect and normalize inconsistent representations. Detection only — normalization is a separate remediation step.*

- **Categorical Representation Consistency** — Same concept, different encodings.
  - *Example:* `Male`, `male`, `M`, `1` all representing the same category.
  - *Sub:* Category mapping consistency across related columns — `country_code = 'US'` but `country_name = 'Canada'`.

- **Date Format Consistency** — Dates stored in non-uniform formats.
  - *Example:* `2024-01-15`, `01/15/2024`, `15-Jan-2024` co-existing in one column.

- **Numeric Format Consistency** — Numbers stored with embedded symbols, separators, or units.
  - *Example:* `"$1,200.50"`, `"1.200,50 EUR"`, `"5 kg"` in a numeric column.

- **Boolean Representation Consistency** — Mixed boolean encodings co-existing.
  - *Example:* `Yes/No`, `True/False`, `1/0`, `Y/N` all present in one column.

- **Text Normalization Issues**
  - *Sub:* Leading/trailing whitespace — `" John "` vs `"John"`.
  - *Sub:* Multiple consecutive whitespaces — `"John    Doe"`.
  - *Sub:* Inconsistent capitalization — `"New York"`, `"new york"`, `"NEW YORK"`.
  - *Sub:* Escape/control characters — embedded `\n`, `\t`, `\r`, `\0`, non-printable characters that corrupt parsing.

- **Null Representation Consistency** — Non-standard nulls masquerading as data.
  - *Example:* `"N/A"`, `"NULL"`, `"None"`, `""`, `" "`, `"—"`, `"unknown"`, `"TBD"`, `"."`, `"NIL"`.
  - *Note:* This check only **detects and normalizes**. Counting missingness happens in 6.4.

---

### 6.4) COMPLETENESS CHECKS
*Measure missingness after null normalization. Understand patterns and impact.*

- **Missingness Measurement**
  - *Sub:* Per column — null percentage per column.
  - *Sub:* Per row — rows with excessive missing values (e.g., >50% null).
  - *Sub:* Per cell — total null rate after normalizing variants from 6.3.

- **Missingness Pattern Analysis**
  - *Sub:* Randomness classification — MCAR / MAR / MNAR.
  - *Sub:* Missing value correlation — missingness in `income` correlated with missingness in `employment_status`.
  - *Sub:* Temporal gaps — missing time periods in time-series data (e.g., no records for Q3 2023).

- **Critical Field Coverage** — Required fields (per business rules) must be non-null.
  - *Example:* `customer_id` must never be null.

---

### 6.5) UNIQUENESS CHECKS
*Ensure no duplicate data and each record is uniquely identifiable.*

- **Row-Level Uniqueness** — Entire rows duplicated.
  - *Example:* Two identical rows for `customer_id = 101` with all columns matching.

- **Key-Level Uniqueness** — Primary/unique key columns contain no duplicates.
  - *Example:* `order_id` appears twice.

- **Composite Key Uniqueness** — Multi-column keys are unique together.
  - *Example:* `(store_id, product_id, date)` should be unique; found duplicate.

- **Distribution Uniqueness** — Distinct value counts in categorical columns (too few/too many).
  - *Example:* `state` has 200 distinct values in a US-only dataset (expected ≤ 50) → likely encoding inconsistency (cross-reference 6.3).
  - *Example:* `country` has 1 distinct value in a global dataset → suspiciously low.

---

### 6.6) CROSS-COLUMN CONSISTENCY CHECKS
*Validate logical relationships and contradictions across columns within a row.*

- **Logical Relationship Validation** — Derived columns match their source columns.
  - *Example:* `full_name = fname + ' ' + lname` must hold.
  - *Example:* `total = subtotal + tax`.

- **Logical Contradiction Detection** — Values that conflict logically.
  - *Example:* `age < 18` AND `job_title = 'CEO'`.
  - *Example:* `marital_status = 'Single'` AND `spouse_name` is populated.

- **Cross-Column Range Logic** — Ranges that don't make sense together.
  - *Example:* `birth_date > hire_date`.
  - *Example:* `end_date < start_date`.
  - *Example:* `death_date < birth_date`.

- **Dependency Rule Validation** — Conditional required/forbidden fields.
  - *Example:* `employee_status = 'Terminated'` but `end_date` is null.
  - *Example:* `payment_method = 'Credit Card'` but `card_number` is null.

- **Category Mapping Consistency** — Codes and descriptions align across related columns.
  - *Example:* `country_code = 'US'` but `country_name = 'Canada'`.
  - *Example:* `dept_code = 'ENG'` but `dept_name = 'Sales'`.

---

### 6.7) REFERENTIAL INTEGRITY CHECKS
*Ensure relationships between tables are valid. One check, three views.*

- **Foreign Key Validity** — Every FK value exists in the parent table's PK.
  - *View 1 — Orphan records:* Child row's FK has no matching parent.
    - *Example:* `orders.customer_id = 999` but no `customers.id = 999`.
  - *View 2 — Missing parent records:* Parent row absent for an existing child reference.
    - *Example:* All orders for `customer_id = 500` exist, but customer 500 was deleted.
  - *View 3 — Invalid FK format:* FK value itself is malformed.
    - *Example:* `customer_id = "ABC"` in a numeric FK column.

- **Lookup Table Conformance** — Referenced values exist in lookup/reference tables.
  - *Example:* `orders.status_code = 'X9'` not present in `status_lookup`.

- **Cross-Table Consistency** — Aggregates reconcile across tables.
  - *Example:* `SUM(orders.amount)` per customer ≠ `customers.total_spend`.

---

### 6.8) BUSINESS RULE & ACCURACY CHECKS
*Domain-specific rules and statistical anomalies. All business rules live here — nowhere else.*

- **Business Rule Violations** — Values that break defined business constraints.
  - *Example:* `salary > 1,000,000` for a non-executive role.
  - *Example:* `discount_percent > 100`.
  - *Example:* `order_quantity > max_order_limit`.

- **Statistical Outlier Detection** — Values deviating significantly from mean/median.
  - *Example:* `age = 200` in a customer dataset (Z-score > 5).
  - *Example:* `transaction_amount` in the 99.99th percentile.

- **Distribution Anomalies** — Imbalances, sudden spikes/drops.
  - *Example:* Daily order count drops 90% on a single day with no holiday.
  - *Example:* `gender` distribution 99% `M` in a general population dataset.

---

### 6.9) TIMELINESS CHECKS
*Monitor data freshness, latency, and temporal logic. Only time-related checks live here.*

- **Data Freshness** — Age of data and last update timestamp.
  - *Example:* `last_updated` is 45 days old for a daily-refresh table.

- **Load Timestamp Analysis** — Consistency of ETL/load timestamps.
  - *Example:* `load_timestamp` < `source_timestamp` (loaded before generated).
  - *Example:* Load timestamps out of order across partitions.

- **Source-to-Target Latency** — Delay between source generation and target availability.
  - *Example:* Source event at 10:00, loaded at 14:00 → 4-hour latency exceeds 1-hour SLA.

- **Temporal Consistency** — Timestamps are logical.
  - *Example:* Future-dated `created_at`.
  - *Example:* `updated_at` before `created_at`.
  - *Example:* Data older than retention policy.

- **Temporal Gaps** — Missing time periods in time-series data.
  - *Note:* If treated as a **completeness** issue, keep in 6.4 only. Recommended home: **6.4** (missingness) with a cross-reference here for monitoring freshness cadence. Choose one and stick to it.

In [0]:
# 6.1.1) Column Presence & Order
# ─────────────────────────────────────────────────────────────────────
# Expected columns exist; no missing or unexpected columns;
# order matches schema contract.

print("=" * 80)
print("6.1.1) COLUMN PRESENCE & ORDER")
print("    Verifying expected columns exist, no extras, and correct order")
print("=" * 80)

# ══ Expected schema contract for raw yelp_tip ══
expected_columns = ["user_id", "business_id", "text", "date", "compliment_count"]
actual_columns   = df.columns

print(f"\n   Expected columns ({len(expected_columns)}): {expected_columns}")
print(f"   Actual columns   ({len(actual_columns)}): {actual_columns}")

findings = []

# ── 1) Missing columns ──
missing = [c for c in expected_columns if c not in actual_columns]
if missing:
    print(f"\n   ⚠️  MISSING columns (in contract but not in data): {missing}")
else:
    print(f"\n   ✅ No missing columns — all expected columns are present")

# ── 2) Unexpected / extra columns ──
unexpected = [c for c in actual_columns if c not in expected_columns]
if unexpected:
    print(f"   ⚠️  UNEXPECTED columns (in data but not in contract): {unexpected}")
else:
    print(f"   ✅ No unexpected columns — no extras beyond the contract")

# ── 3) Column order ──
if list(actual_columns) == expected_columns:
    print(f"   ✅ Column order matches the expected schema contract")
else:
    print(f"   ⚠️  Column order differs from expected schema contract")
    max_len = max(len(expected_columns), len(actual_columns))
    for i in range(max_len):
        exp = expected_columns[i] if i < len(expected_columns) else "<N/A>"
        act = actual_columns[i]   if i < len(actual_columns)   else "<N/A>"
        marker = "✅" if exp == act else "⚠️ "
        print(f"      Position {i}: expected '{exp}', got '{act}' {marker}")

findings.append({"Check": "Missing Columns",    "Detail": str(missing) if missing else "None",       "Status": "PASS" if not missing else "FINDING"})
findings.append({"Check": "Unexpected Columns",  "Detail": str(unexpected) if unexpected else "None", "Status": "PASS" if not unexpected else "FINDING"})
findings.append({"Check": "Column Order",        "Detail": "Matches contract" if list(actual_columns) == expected_columns else "Order differs", "Status": "PASS" if list(actual_columns) == expected_columns else "FINDING"})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
if violation_count == 0:
    print("\n✅ Column presence and order match the schema contract perfectly.")
else:
    print(f"\n⚠️  {violation_count} issue(s) detected — review above.")
print("COLUMN PRESENCE & ORDER CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.1.2) Data Type Conformance
# ─────────────────────────────────────────────────────────────────────
# Check the column's declared (schema) type matches the type specified
# in the schema contract. Detects widening / narrowing / precision mismatches.

print("=" * 80)
print("6.1.2) DATA TYPE CONFORMANCE")
print("    Checking declared types match the schema contract")
print("=" * 80)

# ══ Expected type contract (raw yelp_tip) ══
expected_types = {
    "user_id":           "string",
    "business_id":       "string",
    "text":              "string",
    "date":              "string",   # stored as single timestamp string YYYY-MM-DD HH:MM:SS
    "compliment_count":  "bigint",
}

actual_types = dict(df.dtypes)
findings = []

# Type relationship helpers
widening_pairs  = {("int", "bigint"), ("float", "double"), ("date", "timestamp"), ("int", "double"), ("int", "float")}
narrowing_pairs = {(v, k) for k, v in widening_pairs}

for col_name, expected_type in expected_types.items():
    actual_type = actual_types.get(col_name, "COLUMN MISSING")
    match = (actual_type == expected_type)
    icon   = "✅" if match else "⚠️ "
    status = "PASS" if match else "FINDING"

    print(f"\n   {icon} {col_name}:")
    print(f"      Contract type: {expected_type}")
    print(f"      Actual type:   {actual_type}")
    print(f"      Status:        {status}")

    if not match and actual_type != "COLUMN MISSING":
        pair = (expected_type, actual_type)
        if pair in widening_pairs:
            print(f"      ℹ️  Widening mismatch — actual is a broader type (may be acceptable)")
        elif pair in narrowing_pairs:
            print(f"      ⚠️  Narrowing mismatch — actual is narrower (potential data loss)")
        else:
            print(f"      ⚠️  Type mismatch — incompatible types")

    findings.append({
        "Column": col_name,
        "Expected Type": expected_type,
        "Actual Type": actual_type,
        "Match": "Yes" if match else "No",
        "Status": status
    })

# ── Extra columns not in contract ──
extra_cols = [c for c in actual_types if c not in expected_types]
if extra_cols:
    print(f"\n   ℹ️  Extra columns not in schema contract:")
    for c in extra_cols:
        print(f"      + {c}: {actual_types[c]}")
        findings.append({"Column": c, "Expected Type": "N/A (not in contract)", "Actual Type": actual_types[c], "Match": "N/A", "Status": "INFO"})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
if violation_count == 0:
    print("\n✅ All column types match the schema contract.")
else:
    print(f"\n⚠️  {violation_count} type mismatch(es) detected — review above.")
print("DATA TYPE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.1.3) Semantic Data Type Violations
# Instead of comparing schema types against themselves (which is circular and always passes),
# this infers the SEMANTIC type from actual data values and flags mismatches with the stored type.

from pyspark.sql.types import (
    IntegerType, LongType, StringType, DoubleType, FloatType,
    DateType, TimestampType, BooleanType, ArrayType, MapType, StructType
)
from datetime import datetime as _dt

print("=" * 80)
print("6.1.3) SEMANTIC DATA TYPE VIOLATIONS")
print("    Infers the likely intended type from actual values vs. stored schema type")
print("=" * 80)

total_rows = df.count()
findings = []

for field in df.schema.fields:
    col_name = field.name
    actual_type = field.dataType
    actual_type_str = actual_type.simpleString()

    non_null_df = df.select(col(col_name)).filter(col(col_name).isNotNull())
    non_null_count = non_null_df.count()
    sample_values = [row[0] for row in non_null_df.distinct().limit(10).collect()]

    violations = []
    inferred_type = actual_type_str  # default: type is correct

    # ── Detect nested / complex types ──
    if isinstance(actual_type, ArrayType):
        print(f"\n{'─'*60}")
        print(f"📌 Column: {col_name}")
        print(f"   🔎 Nested: ArrayType — element: {actual_type.elementType}")
    elif isinstance(actual_type, MapType):
        print(f"\n{'─'*60}")
        print(f"📌 Column: {col_name}")
        print(f"   🔎 Nested: MapType — key: {actual_type.keyType}, value: {actual_type.valueType}")
    elif isinstance(actual_type, StructType):
        sub = [f"{sf.name}: {sf.dataType.simpleString()}" for sf in actual_type.fields]
        print(f"\n{'─'*60}")
        print(f"📌 Column: {col_name}")
        print(f"   🔎 Nested: StructType — fields: {', '.join(sub)}")

    # ════════════════════════════════════════════════════════════
    # STRING columns: probe what the values actually represent
    # ════════════════════════════════════════════════════════════
    if isinstance(actual_type, StringType) and non_null_count > 0:

        avg_length = non_null_df.select(mean(length(col(col_name))).alias("avg_len")).collect()[0]["avg_len"]
        min_length = non_null_df.select(spark_min(length(col(col_name)))).collect()[0][0]
        max_length = non_null_df.select(spark_max(length(col(col_name)))).collect()[0][0]

        yelp_id_ratio = non_null_df.filter(col(col_name).rlike("^[A-Za-z0-9_-]{22}$")).count() / non_null_count
        ts_like_ratio = non_null_df.filter(col(col_name).rlike("^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$")).count() / non_null_count
        date_like_ratio = non_null_df.filter(col(col_name).rlike("^\\d{4}-\\d{2}-\\d{2}$")).count() / non_null_count
        year_month_ratio = non_null_df.filter(col(col_name).rlike("^\\d{4}-\\d{2}$")).count() / non_null_count
        year_quarter_ratio = non_null_df.filter(col(col_name).rlike("^\\d{4}-Q[1-4]$")).count() / non_null_count

        month_name_values = ['january','february','march','april','may','june','july','august','september','october','november','december']
        month_abbr_values = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
        dow_name_values = ['sunday','monday','tuesday','wednesday','thursday','friday','saturday']
        dow_abbr_values = ['sun','mon','tue','wed','thu','fri','sat']
        quarter_name_values = ['q1','q2','q3','q4']
        half_year_name_values = ['h1','h2']
        time_of_day_values = ['morning','afternoon','evening','night']
        am_pm_values = ['am','pm']

        month_name_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(month_name_values)).count() / non_null_count
        month_abbr_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(month_abbr_values)).count() / non_null_count
        dow_name_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(dow_name_values)).count() / non_null_count
        dow_abbr_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(dow_abbr_values)).count() / non_null_count
        quarter_name_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(quarter_name_values)).count() / non_null_count
        half_year_name_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(half_year_name_values)).count() / non_null_count
        time_of_day_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(time_of_day_values)).count() / non_null_count
        am_pm_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(am_pm_values)).count() / non_null_count

        json_count = non_null_df.filter(
            (trim(col(col_name)).startswith("{") & trim(col(col_name)).endswith("}")) |
            (trim(col(col_name)).startswith("[") & trim(col(col_name)).endswith("]"))
        ).count()
        json_ratio = json_count / non_null_count

        bool_patterns = ['true','false','yes','no','0','1','y','n','t','f']
        bool_ratio = non_null_df.filter(lower(trim(col(col_name))).isin(bool_patterns)).count() / non_null_count
        numeric_ratio = non_null_df.filter(col(col_name).rlike("^-?\\d+(\\.\\d+)?$")).count() / non_null_count

        if yelp_id_ratio > 0.95:
            inferred_type = "yelp_identifier"
        elif ts_like_ratio > 0.95:
            inferred_type = "timestamp_string"
            violations.append(f"🔶 {ts_like_ratio*100:.1f}% match yyyy-MM-dd HH:mm:ss")
            violations.append("   → Recommended: cast to TimestampType")
        elif date_like_ratio > 0.95:
            inferred_type = "date_string"
            violations.append(f"🔶 {date_like_ratio*100:.1f}% match yyyy-MM-dd")
            violations.append("   → Recommended: cast to DateType")
        elif year_month_ratio > 0.95:
            inferred_type = "year_month_label"
        elif year_quarter_ratio > 0.95:
            inferred_type = "year_quarter_label"
        elif month_name_ratio > 0.95:
            inferred_type = "month_name"
        elif month_abbr_ratio > 0.95:
            inferred_type = "month_abbreviation"
        elif dow_name_ratio > 0.95:
            inferred_type = "day_of_week_name"
        elif dow_abbr_ratio > 0.95:
            inferred_type = "day_of_week_abbreviation"
        elif quarter_name_ratio > 0.95:
            inferred_type = "quarter_label"
        elif half_year_name_ratio > 0.95:
            inferred_type = "half_year_label"
        elif time_of_day_ratio > 0.95:
            inferred_type = "time_of_day_bucket"
        elif am_pm_ratio > 0.95:
            inferred_type = "am_pm_label"
        elif json_ratio > 0.50:
            inferred_type = "struct / map (JSON)"
            violations.append(f"🔶 {json_ratio*100:.1f}% of values appear to be JSON")
            violations.append("   → Recommended: parse with from_json() / schema_of_json()")
        elif bool_ratio > 0.95:
            inferred_type = "boolean"
            violations.append(f"🔶 {bool_ratio*100:.1f}% match boolean patterns")
            violations.append("   → Recommended: cast to BooleanType")
        elif avg_length is not None and avg_length > 40:
            inferred_type = "free_text"
        elif numeric_ratio > 0.95:
            castable_long = non_null_df.filter(col(col_name).rlike("^-?\\d+$")).count() / non_null_count
            if castable_long > 0.95:
                inferred_type = "long / integer"
                violations.append(f"🔶 {numeric_ratio*100:.1f}% castable to numeric (integer-like)")
                violations.append("   → Recommended: cast to LongType or IntegerType")
            else:
                inferred_type = "double / float"
                violations.append(f"🔶 {numeric_ratio*100:.1f}% castable to numeric (decimal)")
                violations.append("   → Recommended: cast to DoubleType")
        elif min_length == max_length and max_length is not None and max_length <= 5:
            inferred_type = f"fixed_width_string(len={max_length})"
        else:
            inferred_type = "categorical_string"

    # ════════════════════════════════════════════════════════════
    # INTEGER / LONG columns: detect boolean-like or constant flags
    # ════════════════════════════════════════════════════════════
    elif isinstance(actual_type, (IntegerType, LongType)) and non_null_count > 0:
        min_val = df.agg(spark_min(col(col_name))).collect()[0][0]
        max_val = df.agg(spark_max(col(col_name))).collect()[0][0]
        distinct_count = df.select(countDistinct(col(col_name))).collect()[0][0]

        if distinct_count == 2 and min_val == 0 and max_val == 1:
            inferred_type = "boolean"
            violations.append("ℹ️  Binary (0/1) — could be stored as BooleanType")
        elif distinct_count == 1:
            inferred_type = f"{actual_type_str} (constant)"
            violations.append(f"ℹ️  Column is constant (single value: {min_val})")

    # ════════════════════════════════════════════════════════════
    # DOUBLE / FLOAT columns: detect integer-only values
    # ════════════════════════════════════════════════════════════
    elif isinstance(actual_type, (DoubleType, FloatType)) and non_null_count > 0:
        frac_check = non_null_df.filter(
            (col(col_name) != col(col_name).cast("long").cast(actual_type_str))
        ).count()
        if frac_check == 0:
            inferred_type = "long / integer (no fractional part)"
            violations.append("ℹ️  All values are whole numbers — consider LongType for efficiency")

    # ── Report per column ──
    status = "✅ PASS" if not violations else "⚠️  FINDING"
    print(f"\n{'─'*60}")
    print(f"📌 Column: {col_name}")
    print(f"   Schema Type:    {actual_type_str}")
    print(f"   Inferred Type:  {inferred_type}")
    print(f"   Non-Null Count: {non_null_count:,} / {total_rows:,}")
    print(f"   Status:         {status}")

    if sample_values:
        display_samples = [str(s)[:80] + ("..." if len(str(s)) > 80 else "") for s in sample_values[:3]]
        print(f"   Samples:        {display_samples}")
    for v in violations:
        print(f"   {v}")

    findings.append({
        "Column": col_name,
        "Schema Type": actual_type_str,
        "Inferred Semantic Type": inferred_type,
        "Status": "PASS" if not violations else "FINDING",
        "Detail": "; ".join(v.strip() for v in violations) if violations else "Types aligned"
    })

# ── Summary ──
print(f"\n{'='*80}")
print("SUMMARY")
print("="*80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} columns checked, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All column schema types match their inferred semantic types.")
else:
    print(f"⚠️  {violation_count} column(s) have schema-vs-semantic type mismatches — review above.")
print("SEMANTIC DATA TYPE VIOLATIONS CHECK COMPLETE")
print("=" * 80)

#### For this Comprehensive Assessment to be logical we will need to
1) Fix the Data Types (cast `date` string → timestamp)
2) Extract temporal features from the tip date for downstream analysis

#### Important Note: These fixes won't change anything yet in the original tables, they are just made temporarily here till the cleaning step of the pipeline

In [0]:
# Fix Data Types based on 6.1.3) Semantic Data Type Violations findings
# ─────────────────────────────────────────────────────────────────────
# user_id          : string ✅  (alphanumeric hash IDs — correctly typed)
# business_id      : string ✅  (alphanumeric hash IDs — correctly typed)
# text             : string ✅  (tip body — correctly typed)
# date             : string ⚠️   → cast to timestamp (single YYYY-MM-DD HH:MM:SS)
# compliment_count : bigint ✅  (compliment count — correctly typed)
#
# Important: These fixes are TEMPORARY for this EDA notebook.
# Original tables in bronze_final are NOT modified.

print("=" * 80)
print("FIX DATA TYPES")
print("=" * 80)

print("\n📋 BEFORE fix:")
df.printSchema()

# Cast date string to timestamp
df_typed = (
    df
    .withColumn("tip_timestamp", col("date").cast("timestamp"))  # single timestamp cast
    .drop("date")                                                 # drop original string column
)

print("\n📋 AFTER fix:")
df_typed.printSchema()

# Quick validation: check for null timestamps (failed casts)
null_ts = df_typed.filter(col("tip_timestamp").isNull()).count()
total = df_typed.count()
print(f"\n✅ Rows: {total:,}")
print(f"   Null timestamps (failed casts): {null_ts:,}")

# Spot-check: show a few rows with their typed columns
print("\n📋 Sample (typed):")
display(df_typed.limit(5))

In [0]:
# Remodel: Extract temporal features from tip_timestamp
# (No explode needed — each row is already one tip)
# ─────────────────────────────────────────────────────────────────────
from pyspark.sql.functions import (
    hour, minute, second, dayofweek, dayofmonth, dayofyear,
    weekofyear, month, year, quarter, date_format, last_day,
    next_day, trunc, months_between, lit, when, floor, ceil, concat
)

print("=" * 80)
print("REMODEL DATA  —  Full Date-Dimension Extraction")
print("=" * 80)

original_rows = df_typed.count()
original_cols = len(df_typed.columns)

df_remodeled = (
    df_typed

    # ════════════════════════════════════════════════════════════════════
    # DATE-LEVEL ATTRIBUTES  (date dimension style)
    # ════════════════════════════════════════════════════════════════════
    .withColumn("tip_date",        col("tip_timestamp").cast("date"))
    .withColumn("tip_year",        year(col("tip_timestamp")))
    .withColumn("tip_month",       month(col("tip_timestamp")))
    .withColumn("tip_month_name",  date_format(col("tip_timestamp"), "MMMM"))        # January…
    .withColumn("tip_month_abbr",  date_format(col("tip_timestamp"), "MMM"))          # Jan…
    .withColumn("tip_day",         dayofmonth(col("tip_timestamp")))
    .withColumn("tip_day_of_year", dayofyear(col("tip_timestamp")))
    .withColumn("tip_dow_number",  dayofweek(col("tip_timestamp")))                   # 1=Sun…7=Sat
    .withColumn("tip_dow_name",    date_format(col("tip_timestamp"), "EEEE"))         # Monday…
    .withColumn("tip_dow_abbr",    date_format(col("tip_timestamp"), "EEE"))          # Mon…
    .withColumn("tip_quarter",     quarter(col("tip_timestamp")))
    .withColumn("tip_quarter_name",concat(lit("Q"), quarter(col("tip_timestamp"))))
    .withColumn("tip_week_of_year",weekofyear(col("tip_timestamp")))
    .withColumn("tip_year_month",  date_format(col("tip_timestamp"), "yyyy-MM"))      # 2019-07
    .withColumn("tip_year_quarter",
        concat(year(col("tip_timestamp")).cast("string"), lit("-Q"), quarter(col("tip_timestamp")).cast("string"))
    )

    # ── Half-year / Semester ──
    .withColumn("tip_half_year",
        when(month(col("tip_timestamp")) <= 6, lit(1)).otherwise(lit(2))
    )
    .withColumn("tip_half_year_name",
        when(month(col("tip_timestamp")) <= 6, lit("H1")).otherwise(lit("H2"))
    )

    # ── Month position flags ──
    .withColumn("tip_month_start_date", trunc(col("tip_timestamp"), "month"))
    .withColumn("tip_month_end_date",   last_day(col("tip_timestamp")))
    .withColumn("is_month_start",
        (dayofmonth(col("tip_timestamp")) == 1).cast("boolean")
    )
    .withColumn("is_month_end",
        (col("tip_date") == last_day(col("tip_timestamp"))).cast("boolean")
    )

    # ── Year position flags ──
    .withColumn("tip_year_start_date", trunc(col("tip_timestamp"), "year"))
    .withColumn("is_year_start",
        ((month(col("tip_timestamp")) == 1) & (dayofmonth(col("tip_timestamp")) == 1)).cast("boolean")
    )
    .withColumn("is_year_end",
        ((month(col("tip_timestamp")) == 12) & (dayofmonth(col("tip_timestamp")) == 31)).cast("boolean")
    )

    # ── Weekend / Weekday flag ──
    .withColumn("is_weekend",
        dayofweek(col("tip_timestamp")).isin(1, 7).cast("boolean")  # 1=Sun, 7=Sat
    )
    .withColumn("is_weekday",
        (~dayofweek(col("tip_timestamp")).isin(1, 7)).cast("boolean")
    )

    # ════════════════════════════════════════════════════════════════════
    # TIME-LEVEL ATTRIBUTES
    # ════════════════════════════════════════════════════════════════════
    .withColumn("tip_hour",   hour(col("tip_timestamp")))
    .withColumn("tip_minute", minute(col("tip_timestamp")))
    .withColumn("tip_second", second(col("tip_timestamp")))
    .withColumn("tip_time_of_day",
        when(hour(col("tip_timestamp")).between(5, 11),  lit("Morning"))
        .when(hour(col("tip_timestamp")).between(12, 16), lit("Afternoon"))
        .when(hour(col("tip_timestamp")).between(17, 20), lit("Evening"))
        .otherwise(lit("Night"))
    )
    .withColumn("tip_am_pm",
        when(hour(col("tip_timestamp")) < 12, lit("AM")).otherwise(lit("PM"))
    )
    .withColumn("tip_hour_12",
        when(hour(col("tip_timestamp")) == 0, lit(12))
        .when(hour(col("tip_timestamp")) > 12, hour(col("tip_timestamp")) - 12)
        .otherwise(hour(col("tip_timestamp")))
    )
    .withColumn("tip_hour_bin_4h",
        (floor(hour(col("tip_timestamp")) / 4) * 4).cast("int")
    )
)

# Reassign df and total_rows so ALL downstream cells use the remodeled data
df = df_remodeled
total_rows = df.count()

print(f"\n📊 Remodeled shape: {total_rows:,} rows × {len(df.columns)} columns")
print(f"   (Original: {original_rows:,} rows × {original_cols} cols → same row count + temporal features)")
print("\n📋 Schema:")
df.printSchema()
print("📋 Sample (remodeled):")
display(df.limit(10))

In [0]:
# 6.1.4) Schema Drift Detection
# ─────────────────────────────────────────────────────────────────────
# Compare current remodeled schema against expected baseline;
# flag added / removed / retyped columns.

print("=" * 80)
print("6.1.4) SCHEMA DRIFT DETECTION")
print("    Comparing current schema against expected remodeled baseline")
print("=" * 80)

# ══ Expected post-remodel baseline ══
baseline_schema = {
    # Original tip columns (after type fixes)
    "user_id": "string",
    "business_id": "string",
    "text": "string",
    "compliment_count": "bigint",
    "tip_timestamp": "timestamp",
    # Derived temporal features
    "tip_date": "date",
    "tip_year": "int", "tip_month": "int", "tip_day": "int",
    "tip_month_name": "string", "tip_month_abbr": "string",
    "tip_day_of_year": "int", "tip_dow_number": "int",
    "tip_dow_name": "string", "tip_dow_abbr": "string",
    "tip_quarter": "int", "tip_quarter_name": "string",
    "tip_week_of_year": "int",
    "tip_year_month": "string", "tip_year_quarter": "string",
    "tip_half_year": "int", "tip_half_year_name": "string",
    "tip_month_start_date": "date", "tip_month_end_date": "date",
    "is_month_start": "boolean", "is_month_end": "boolean",
    "tip_year_start_date": "date",
    "is_year_start": "boolean", "is_year_end": "boolean",
    "is_weekend": "boolean", "is_weekday": "boolean",
    "tip_hour": "int", "tip_minute": "int", "tip_second": "int",
    "tip_time_of_day": "string", "tip_am_pm": "string",
    "tip_hour_12": "int", "tip_hour_bin_4h": "int",
}

current_schema = dict(df.dtypes)
findings = []

# 1) Added columns
added = {c: current_schema[c] for c in current_schema if c not in baseline_schema}
if added:
    print(f"\n   ⚠️  ADDED columns (not in baseline):")
    for c, t in added.items():
        print(f"      + {c}: {t}")
else:
    print(f"\n   ✅ No new columns added beyond baseline")

# 2) Removed columns
removed = {c: baseline_schema[c] for c in baseline_schema if c not in current_schema}
if removed:
    print(f"\n   ⚠️  REMOVED columns (missing from current):")
    for c, t in removed.items():
        print(f"      - {c}: {t}")
else:
    print(f"   ✅ No columns removed from baseline")

# 3) Retyped columns
retyped = {}
for c in baseline_schema:
    if c in current_schema and baseline_schema[c] != current_schema[c]:
        retyped[c] = (baseline_schema[c], current_schema[c])
if retyped:
    print(f"\n   ⚠️  RETYPED columns:")
    for c, (old_t, new_t) in retyped.items():
        print(f"      ~ {c}: {old_t} → {new_t}")
else:
    print(f"   ✅ No columns changed type from baseline")

for c in added:
    findings.append({"Column": c, "Drift": "ADDED",    "Baseline": "N/A",            "Current": current_schema[c], "Status": "FINDING"})
for c in removed:
    findings.append({"Column": c, "Drift": "REMOVED",  "Baseline": baseline_schema[c], "Current": "N/A",            "Status": "FINDING"})
for c, (old_t, new_t) in retyped.items():
    findings.append({"Column": c, "Drift": "RETYPED",  "Baseline": old_t,             "Current": new_t,            "Status": "FINDING"})
for c in baseline_schema:
    if c in current_schema and baseline_schema[c] == current_schema[c]:
        findings.append({"Column": c, "Drift": "UNCHANGED", "Baseline": baseline_schema[c], "Current": current_schema[c], "Status": "PASS"})

print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

drift_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} columns compared, {drift_count} drift(s) detected")
if drift_count == 0:
    print("✅ No schema drift — current schema matches the expected baseline exactly.")
else:
    print(f"⚠️  {drift_count} schema drift(s) detected — review above.")
print("SCHEMA DRIFT DETECTION COMPLETE")
print("=" * 80)

In [0]:
# 6.2.1) Allowed Value Conformance (single column)
# ─────────────────────────────────────────────────────────────────────
# Values belong to the permitted domain/category list.
# Checks all categorical columns derived during remodeling.

print("=" * 80)
print("6.2.1) ALLOWED VALUE CONFORMANCE")
print("    Checking values against closed/allowed sets")
print("=" * 80)

total_rows = df.count()
df_columns = set(df.columns)
findings = []

# ══ Define allowed value sets ══
valid_months_full  = ["January","February","March","April","May","June",
                      "July","August","September","October","November","December"]
valid_months_abbr  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
valid_dow_full     = ["Sunday","Monday","Tuesday","Wednesday","Thursday","Friday","Saturday"]
valid_dow_abbr     = ["Sun","Mon","Tue","Wed","Thu","Fri","Sat"]
valid_time_of_day  = ["Morning","Afternoon","Evening","Night"]
valid_am_pm        = ["AM","PM"]
valid_quarter_name = ["Q1","Q2","Q3","Q4"]
valid_half_year_name = ["H1","H2"]

valid_stars = [1, 2, 3, 4, 5]

categorical_checks = [
    ("tip_month_name",    valid_months_full,    "Full month name"),
    ("tip_month_abbr",    valid_months_abbr,    "Abbreviated month name"),
    ("tip_dow_name",      valid_dow_full,       "Full day-of-week name"),
    ("tip_dow_abbr",      valid_dow_abbr,       "Abbreviated day-of-week name"),
    ("tip_time_of_day",   valid_time_of_day,    "Time of day bucket"),
    ("tip_am_pm",         valid_am_pm,          "AM/PM indicator"),
    ("tip_quarter_name",  valid_quarter_name,   "Quarter name"),
    ("tip_half_year_name",valid_half_year_name,  "Half year name"),
]

for col_name, allowed, desc in categorical_checks:
    if col_name in df_columns:
        bad_count = df.filter(
            ~col(col_name).isin(allowed) & col(col_name).isNotNull()
        ).count()
        status = "PASS" if bad_count == 0 else "FINDING"
        icon = "✅" if bad_count == 0 else "⚠️ "
        print(f"\n   {icon} {col_name}: {bad_count:,} violations — {status}")
        print(f"      Allowed: {allowed}")
        if bad_count > 0:
            bad_vals = (
                df.filter(~col(col_name).isin(allowed) & col(col_name).isNotNull())
                  .select(col_name).distinct().limit(10)
            )
            print(f"      ⚠️  Unexpected values: {[r[0] for r in bad_vals.collect()]}")
        findings.append({
            "Column": col_name,
            "Allowed Values": f"{allowed[:3]}... ({len(allowed)} total)",
            "Description": desc,
            "Violations": bad_count,
            "Status": status
        })

# ── Stars: only 1-5 allowed ──
print(f"\n{'─'*60}")
print("📌 Stars value conformance")
if "stars" in df_columns:
    bad_stars = df.filter(~col("stars").isin(valid_stars) & col("stars").isNotNull()).count()
    status = "PASS" if bad_stars == 0 else "FINDING"
    icon = "✅" if bad_stars == 0 else "⚠️ "
    print(f"   {icon} stars: {bad_stars:,} violations (allowed: {valid_stars}) — {status}")
    if bad_stars > 0:
        bad_vals = [r[0] for r in df.filter(~col("stars").isin(valid_stars) & col("stars").isNotNull()).select("stars").distinct().limit(10).collect()]
        print(f"      ⚠️  Unexpected values: {bad_vals}")
    findings.append({"Column": "stars", "Allowed Values": str(valid_stars), "Description": "Star rating", "Violations": bad_stars, "Status": status})

# ── Boolean columns: only True/False allowed ──
print(f"\n{'─'*60}")
print("📌 Boolean column value conformance")
bool_cols = ["is_month_start", "is_month_end", "is_year_start", "is_year_end", "is_weekend", "is_weekday"]
for col_name in bool_cols:
    if col_name in df_columns:
        null_count = df.filter(col(col_name).isNull()).count()
        status = "PASS" if null_count == 0 else "FINDING"
        icon = "✅" if null_count == 0 else "⚠️ "
        print(f"   {icon} {col_name}: {null_count:,} NULL values (boolean must be True/False) — {status}")
        findings.append({"Column": col_name, "Allowed Values": "True, False", "Description": "Boolean flag", "Violations": null_count, "Status": status})

print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} category checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All categorical and boolean columns contain only valid/allowed values.")
else:
    print(f"⚠️  {violation_count} column(s) have invalid categories — review above.")
print("ALLOWED VALUE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.2.2) Range Conformance (single column)
# ─────────────────────────────────────────────────────────────────────
# Values within min/max bounds for numeric/temporal columns.

print("=" * 80)
print("6.2.2) RANGE CONFORMANCE")
print("    Checking numeric/temporal values fall within physically valid bounds")
print("=" * 80)

total_rows = df.count()
df_columns = set(df.columns)
findings = []

# ══ Define valid ranges: (column, min, max, description) ══
range_checks = [
    # Tip-specific numeric columns
    ("compliment_count",     0, None, "Compliment count (non-negative)"),
    # Temporal columns
    ("tip_year",          1950, 2026, "Year"),
    ("tip_month",         1, 12,   "Month"),
    ("tip_day",           1, 31,   "Day of month"),
    ("tip_day_of_year",   1, 366,  "Day of year"),
    ("tip_dow_number",    1, 7,    "Day of week (1=Sun..7=Sat)"),
    ("tip_quarter",       1, 4,    "Quarter"),
    ("tip_week_of_year",  1, 53,   "Week of year"),
    ("tip_half_year",     1, 2,    "Half year"),
    ("tip_hour",          0, 23,   "Hour"),
    ("tip_minute",        0, 59,   "Minute"),
    ("tip_second",        0, 59,   "Second"),
    ("tip_hour_12",       1, 12,   "Hour (12h format)"),
    ("tip_hour_bin_4h",   0, 20,   "Hour bin (4h buckets: 0,4,8,12,16,20)"),
]

for col_name, min_val, max_val, desc in range_checks:
    if col_name in df_columns:
        # Handle unbounded upper range (None means no upper bound)
        if max_val is not None:
            violation_condition = (col(col_name) < min_val) | (col(col_name) > max_val)
            display_max = max_val
        else:
            violation_condition = col(col_name) < min_val
            display_max = "∞"

        out_of_range = df.filter(violation_condition).count()
        status = "PASS" if out_of_range == 0 else "FINDING"
        icon = "✅" if out_of_range == 0 else "⚠️ "
        print(f"   {icon} {col_name} [{min_val}–{display_max}]: {out_of_range:,} violations — {status}")

        if out_of_range > 0:
            bad_vals = (
                df.filter(violation_condition)
                  .select(col_name).distinct().limit(10)
            )
            print(f"      Out-of-range values: {[r[0] for r in bad_vals.collect()]}")

        findings.append({
            "Column": col_name,
            "Valid Range": f"[{min_val}–{display_max}]",
            "Description": f"{desc} within valid range",
            "Violations": out_of_range,
            "Status": status
        })

# ── Timestamp plausibility (Yelp founded 2004, data should be ≤ today) ──
print(f"\n{'─'*60}")
print("📌 Timestamp plausibility (Yelp founded 2004, data should be ≤ today)")

if "tip_timestamp" in df_columns:
    from pyspark.sql.functions import min as spark_min, max as spark_max

    ts_stats = df.agg(
        spark_min("tip_timestamp").alias("min_ts"),
        spark_max("tip_timestamp").alias("max_ts")
    ).collect()[0]
    print(f"   Earliest tip: {ts_stats['min_ts']}")
    print(f"   Latest tip:   {ts_stats['max_ts']}")

    implausible = df.filter(
        (year(col("tip_timestamp")) < 2004) |
        (col("tip_timestamp") > current_date())
    ).count()

    status = "PASS" if implausible == 0 else "FINDING"
    icon = "✅" if implausible == 0 else "⚠️ "
    print(f"   {icon} Implausible timestamps (before 2004 or future): {implausible:,} — {status}")

    findings.append({
        "Column": "tip_timestamp",
        "Valid Range": "[2004 – today]",
        "Description": "Timestamp within Yelp's operational period",
        "Violations": implausible,
        "Status": status
    })

print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} range checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All numeric/temporal values fall within valid bounds.")
else:
    print(f"⚠️  {violation_count} column(s) have range violations — review above.")
print("RANGE CONFORMANCE CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.2.3) Pattern Conformance
# ─────────────────────────────────────────────────────────────────────
# Values match required format (regex patterns, expected structure).

from pyspark.sql.functions import length, col, when, lit, sum as spark_sum

print("=" * 80)
print("6.2.3) PATTERN CONFORMANCE")
print("    Checking format violations across all columns")
print("=" * 80)

total_rows = df.count()
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 1) user_id — 22-char alphanumeric hash (Yelp format)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 1: user_id — 22-char alphanumeric hash pattern")
print("   Expected pattern: ^[a-zA-Z0-9_-]{22}$")

bad_rid_count = df.filter(~col("user_id").rlike("^[a-zA-Z0-9_-]{22}$")).count()
if bad_rid_count == 0:
    print(f"   ✅ PASS — All {total_rows:,} rows have valid 22-char alphanumeric user_id")
else:
    print(f"   ⚠️  FINDING — {bad_rid_count:,} / {total_rows:,} rows have non-conforming user_id")
    bad_samples = [row[0] for row in df.filter(~col("user_id").rlike("^[a-zA-Z0-9_-]{22}$")).select("user_id").distinct().limit(10).collect()]
    print(f"   Examples: {bad_samples}")
findings.append({"Column": "user_id", "Pattern": "^[a-zA-Z0-9_-]{22}$", "Description": "22-char alphanumeric Yelp hash", "Violations": bad_rid_count, "Status": "PASS" if bad_rid_count == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 2) user_id — 22-char alphanumeric hash (Yelp format)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 2: user_id — 22-char alphanumeric hash pattern")
print("   Expected pattern: ^[a-zA-Z0-9_-]{22}$")

bad_uid_count = df.filter(~col("user_id").rlike("^[a-zA-Z0-9_-]{22}$")).count()
if bad_uid_count == 0:
    print(f"   ✅ PASS — All {total_rows:,} rows have valid 22-char alphanumeric user_id")
else:
    print(f"   ⚠️  FINDING — {bad_uid_count:,} / {total_rows:,} rows have non-conforming user_id")
    bad_samples = [row[0] for row in df.filter(~col("user_id").rlike("^[a-zA-Z0-9_-]{22}$")).select("user_id").distinct().limit(10).collect()]
    print(f"   Examples: {bad_samples}")
findings.append({"Column": "user_id", "Pattern": "^[a-zA-Z0-9_-]{22}$", "Description": "22-char alphanumeric Yelp hash", "Violations": bad_uid_count, "Status": "PASS" if bad_uid_count == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 3) business_id — 22-char alphanumeric hash (Yelp format)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 3: business_id — 22-char alphanumeric hash pattern")
print("   Expected pattern: ^[a-zA-Z0-9_-]{22}$")

bad_bid_count = df.filter(~col("business_id").rlike("^[a-zA-Z0-9_-]{22}$")).count()
if bad_bid_count == 0:
    print(f"   ✅ PASS — All {total_rows:,} rows have valid 22-char alphanumeric business_id")
else:
    print(f"   ⚠️  FINDING — {bad_bid_count:,} / {total_rows:,} rows have non-conforming business_id")
    bad_samples = [row[0] for row in df.filter(~col("business_id").rlike("^[a-zA-Z0-9_-]{22}$")).select("business_id").distinct().limit(10).collect()]
    print(f"   Examples: {bad_samples}")
findings.append({"Column": "business_id", "Pattern": "^[a-zA-Z0-9_-]{22}$", "Description": "22-char alphanumeric Yelp hash", "Violations": bad_bid_count, "Status": "PASS" if bad_bid_count == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 4) tip_timestamp — Non-null valid timestamp
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 4: tip_timestamp — Valid timestamp (non-null after cast)")

null_ts_count = df.filter(col("tip_timestamp").isNull()).count()
if null_ts_count == 0:
    print(f"   ✅ PASS — All {total_rows:,} timestamps are valid (no null from failed casts)")
else:
    print(f"   ⚠️  FINDING — {null_ts_count:,} / {total_rows:,} NULL timestamps (malformed source strings)")
findings.append({"Column": "tip_timestamp", "Pattern": "Valid ISO 8601 timestamp", "Description": "Non-null after cast", "Violations": null_ts_count, "Status": "PASS" if null_ts_count == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 5) tip_date — Consistent with timestamp
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 5: tip_date — Consistent with tip_timestamp")

date_mismatch = df.filter(
    col("tip_date") != col("tip_timestamp").cast("date")
).count()
if date_mismatch == 0:
    print(f"   ✅ PASS — All {total_rows:,} tip_date values match their source timestamp")
else:
    print(f"   ⚠️  FINDING — {date_mismatch:,} rows have date ≠ timestamp.cast('date')")
findings.append({"Column": "tip_date", "Pattern": "== timestamp.cast('date')", "Description": "Date portion matches source timestamp", "Violations": date_mismatch, "Status": "PASS" if date_mismatch == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 6) tip_year_month — yyyy-MM format
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 6: tip_year_month — yyyy-MM pattern")

bad_ym = df.filter(~col("tip_year_month").rlike("^\\d{4}-\\d{2}$")).count()
if bad_ym == 0:
    print(f"   ✅ PASS — All values match yyyy-MM format")
else:
    print(f"   ⚠️  FINDING — {bad_ym:,} values don't match yyyy-MM format")
findings.append({"Column": "tip_year_month", "Pattern": "^\\d{4}-\\d{2}$", "Description": "yyyy-MM format", "Violations": bad_ym, "Status": "PASS" if bad_ym == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 7) tip_year_quarter — yyyy-Qn format
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 7: tip_year_quarter — yyyy-Qn pattern")

bad_yq = df.filter(~col("tip_year_quarter").rlike("^\\d{4}-Q[1-4]$")).count()
if bad_yq == 0:
    print(f"   ✅ PASS — All values match yyyy-Qn format")
else:
    print(f"   ⚠️  FINDING — {bad_yq:,} values don't match yyyy-Qn format")
findings.append({"Column": "tip_year_quarter", "Pattern": "^\\d{4}-Q[1-4]$", "Description": "yyyy-Qn format", "Violations": bad_yq, "Status": "PASS" if bad_yq == 0 else "FINDING"})

# ══════════════════════════════════════════════════════════════════════════════
# 8) tip_hour_bin_4h — Must be one of {0,4,8,12,16,20}
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 Check 8: tip_hour_bin_4h — Valid 4-hour bins")

valid_bins = [0, 4, 8, 12, 16, 20]
bad_bins = df.filter(~col("tip_hour_bin_4h").isin(valid_bins)).count()
if bad_bins == 0:
    print(f"   ✅ PASS — All values are valid 4-hour bins {valid_bins}")
else:
    print(f"   ⚠️  FINDING — {bad_bins:,} values outside valid bins")
findings.append({"Column": "tip_hour_bin_4h", "Pattern": str(valid_bins), "Description": "4-hour bin values", "Violations": bad_bins, "Status": "PASS" if bad_bins == 0 else "FINDING"})

# ══ SUMMARY ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} pattern checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All values conform to their expected patterns and formats.")
else:
    print(f"⚠️  {violation_count} check(s) detected pattern mismatches — review above.")
print("PATTERN CONFORMANCE CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.2.4) Impossible Value Detection
# ─────────────────────────────────────────────────────────────────────
# Values that cannot physically exist regardless of configurable bounds.

print("=" * 80)
print("6.2.4) IMPOSSIBLE VALUE DETECTION")
print("    Checking for values that cannot physically exist")
print("=" * 80)

total_rows = df.count()
findings = []

# ══ 1) Future timestamps ══
print(f"\n{'─'*60}")
print("📌 Check 1: Future timestamps (tip cannot happen in the future)")
future_count = df.filter(col("tip_timestamp") > current_date()).count()
status = "PASS" if future_count == 0 else "FINDING"
icon = "✅" if future_count == 0 else "⚠️ "
print(f"   {icon} Future timestamps: {future_count:,} — {status}")
if future_count > 0:
    samples = df.filter(col("tip_timestamp") > current_date()).select("tip_timestamp").limit(5).collect()
    print(f"   Examples: {[r[0] for r in samples]}")
findings.append({"Check": "Future timestamps", "Description": "tip_timestamp > today", "Violations": future_count, "Status": status})

# ══ 2) Pre-Yelp timestamps (before Oct 2004 when Yelp was founded) ══
print(f"\n{'─'*60}")
print("📌 Check 2: Pre-Yelp timestamps (Yelp founded October 2004)")
pre_yelp = df.filter(col("tip_timestamp") < lit("2004-10-01")).count()
status = "PASS" if pre_yelp == 0 else "FINDING"
icon = "✅" if pre_yelp == 0 else "⚠️ "
print(f"   {icon} Pre-Yelp timestamps (before 2004-10-01): {pre_yelp:,} — {status}")
if pre_yelp > 0:
    samples = df.filter(col("tip_timestamp") < lit("2004-10-01")).select("tip_timestamp").limit(5).collect()
    print(f"   Examples: {[r[0] for r in samples]}")
findings.append({"Check": "Pre-Yelp timestamps", "Description": "tip < 2004-10-01", "Violations": pre_yelp, "Status": status})

# ══ 3) Impossible calendar dates ══
print(f"\n{'─'*60}")
print("📌 Check 3: Impossible calendar dates (e.g., Feb 30, Apr 31)")
null_ts = df.filter(col("tip_timestamp").isNull()).count()
status = "PASS" if null_ts == 0 else "FINDING"
icon = "✅" if null_ts == 0 else "⚠️ "
print(f"   {icon} Null timestamps (impossible date conversions): {null_ts:,} — {status}")
findings.append({"Check": "Impossible calendar dates", "Description": "Null after timestamp cast", "Violations": null_ts, "Status": status})

# ══ 4) Day-of-month > days-in-that-month ══
print(f"\n{'─'*60}")
print("📌 Check 4: Day exceeds days in its month")
bad_dom = df.filter(
    col("tip_day") > dayofmonth(last_day(col("tip_date")))
).count()
status = "PASS" if bad_dom == 0 else "FINDING"
icon = "✅" if bad_dom == 0 else "⚠️ "
print(f"   {icon} Day > max days in month: {bad_dom:,} — {status}")
findings.append({"Check": "Day > days in month", "Description": "tip_day > last_day(month)", "Violations": bad_dom, "Status": status})

# ══ 5) Invalid star ratings ══
print(f"\n{'─'*60}")
print("📌 Check 5: Negative compliment count")
bad_stars = df.filter(col("compliment_count") < 0).count()
status = "PASS" if bad_stars == 0 else "FINDING"
icon = "✅" if bad_stars == 0 else "⚠️ "
print(f"   {icon} Negative compliment_count values: {bad_stars:,} — {status}")
findings.append({"Check": "Negative compliment count", "Description": "compliment_count must be non-negative", "Violations": bad_stars, "Status": status})

# ══ 6) Null compliment count ══
print(f"\n{'─'*60}")
print("📌 Check 6: Null compliment count")
negative_votes = df.filter(col("compliment_count").isNull()).count()
status = "PASS" if negative_votes == 0 else "FINDING"
icon = "✅" if negative_votes == 0 else "⚠️ "
print(f"   {icon} Rows with null compliment_count values: {negative_votes:,} — {status}")
findings.append({"Check": "Null compliment count", "Description": "compliment_count should not be null", "Violations": negative_votes, "Status": status})

# ══ 7) Empty tip text on non-null rows ══
print(f"\n{'─'*60}")
print("📌 Check 7: Empty tip text")
empty_text = df.filter(col("text").isNull() | (trim(col("text")) == "")).count()
status = "PASS" if empty_text == 0 else "FINDING"
icon = "✅" if empty_text == 0 else "⚠️ "
print(f"   {icon} Null or empty review text: {empty_text:,} — {status}")
findings.append({"Check": "Empty tip text", "Description": "text should contain tip content", "Violations": empty_text, "Status": status})

# ══ 8) Contradictory boolean pairs ══
print(f"\n{'─'*60}")
print("📌 Check 8: Contradictory boolean pairs")
both_or_neither = df.filter(
    (col("is_weekend") == col("is_weekday"))
).count()
status = "PASS" if both_or_neither == 0 else "FINDING"
icon = "✅" if both_or_neither == 0 else "⚠️ "
print(f"   {icon} is_weekend == is_weekday (should be opposite): {both_or_neither:,} — {status}")
findings.append({"Check": "Weekend/Weekday contradiction", "Description": "is_weekend == is_weekday", "Violations": both_or_neither, "Status": status})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} impossible-value checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ No physically impossible values detected.")
else:
    print(f"⚠️  {violation_count} check(s) found impossible values — review above.")
print("IMPOSSIBLE VALUE DETECTION COMPLETE")
print("=" * 80)

In [0]:
# 6.3) Format & Encoding Standardization
# ─────────────────────────────────────────────────────────────────────
# Detect inconsistent representations: categorical encoding, text normalization,
# null masquerading as data. Detection only — normalization is a separate step.

from pyspark.sql.functions import regexp_replace, trim, lower, length, col, when, lit

print("=" * 80)
print("6.3) FORMAT & ENCODING STANDARDIZATION")
print("    Detecting inconsistent representations across all columns")
print("=" * 80)

total_rows = df.count()
df_columns = set(df.columns)
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.3.1) Categorical Representation Consistency
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.3.1) Categorical Representation Consistency")
print("   Checking if same concept has different encodings")

string_cat_cols = [
    "tip_month_name", "tip_month_abbr", "tip_dow_name",
    "tip_dow_abbr", "tip_time_of_day", "tip_am_pm",
    "tip_quarter_name", "tip_half_year_name"
]

for col_name in string_cat_cols:
    if col_name in df_columns:
        distinct_vals = [r[0] for r in df.select(col_name).distinct().collect()]
        # Check for case inconsistency: same value in different cases
        lower_set = set(v.lower() if v else "" for v in distinct_vals)
        has_case_inconsistency = len(lower_set) < len(set(distinct_vals))

        icon = "⚠️ " if has_case_inconsistency else "✅"
        status = "FINDING" if has_case_inconsistency else "PASS"
        print(f"   {icon} {col_name}: {len(distinct_vals)} distinct values — {status}")
        if has_case_inconsistency:
            print(f"      Case-inconsistent values detected")
        findings.append({"Check": f"Case consistency: {col_name}", "Distinct Values": len(distinct_vals), "Issue": "Case inconsistency" if has_case_inconsistency else "None", "Status": status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.3.2) Text Normalization Issues
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.3.2) Text Normalization Issues")
print("   Checking for whitespace, control chars, inconsistent casing")

# Check all string columns
string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

for col_name in string_cols:
    issues = []

    # Leading/trailing whitespace
    ws_count = df.filter(col(col_name) != trim(col(col_name))).count()
    if ws_count > 0:
        issues.append(f"leading/trailing whitespace: {ws_count:,}")

    # Multiple consecutive spaces
    multi_space = df.filter(col(col_name).rlike("  +")).count()
    if multi_space > 0:
        issues.append(f"multiple consecutive spaces: {multi_space:,}")

    # Control characters (\n, \t, \r, etc.)
    ctrl_chars = df.filter(col(col_name).rlike("[\\x00-\\x1f\\x7f]")).count()
    if ctrl_chars > 0:
        issues.append(f"control characters: {ctrl_chars:,}")

    # Empty strings (not null but zero-length)
    empty_str = df.filter((col(col_name) == "") | (trim(col(col_name)) == "")).count()
    if empty_str > 0:
        issues.append(f"empty strings: {empty_str:,}")

    if issues:
        print(f"   ⚠️  {col_name}: {'; '.join(issues)}")
        findings.append({"Check": f"Text normalization: {col_name}", "Distinct Values": "-", "Issue": "; ".join(issues), "Status": "FINDING"})
    else:
        print(f"   ✅ {col_name}: clean")
        findings.append({"Check": f"Text normalization: {col_name}", "Distinct Values": "-", "Issue": "None", "Status": "PASS"})

# ══════════════════════════════════════════════════════════════════════════════
# 6.3.3) Null Representation Consistency
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.3.3) Null Representation Consistency")
print("   Checking for non-standard nulls masquerading as data")

null_patterns = ["N/A", "n/a", "NA", "na", "NULL", "null", "None", "none",
                 "NONE", "NaN", "nan", "unknown", "Unknown", "UNKNOWN",
                 "TBD", "tbd", ".", "-", "--", "—", "NIL", "nil", "#N/A", "#VALUE!"]

for col_name in string_cols:
    masked_nulls = df.filter(
        trim(col(col_name)).isin(null_patterns)
    ).count()

    if masked_nulls > 0:
        bad_vals = [r[0] for r in df.filter(trim(col(col_name)).isin(null_patterns)).select(col_name).distinct().collect()]
        print(f"   ⚠️  {col_name}: {masked_nulls:,} masked null values — values: {bad_vals}")
        findings.append({"Check": f"Masked nulls: {col_name}", "Distinct Values": str(bad_vals), "Issue": f"{masked_nulls:,} masked nulls", "Status": "FINDING"})
    else:
        print(f"   ✅ {col_name}: no masked nulls")
        findings.append({"Check": f"Masked nulls: {col_name}", "Distinct Values": "-", "Issue": "None", "Status": "PASS"})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} format/encoding checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All values are consistently formatted with no encoding issues.")
else:
    print(f"⚠️  {violation_count} inconsistency(ies) detected — review above.")
print("FORMAT & ENCODING STANDARDIZATION CHECK COMPLETE")
print("=" * 80)

In [0]:
# 6.4) Completeness Checks
# ─────────────────────────────────────────────────────────────────────
# Measure missingness after null normalization. Understand patterns and impact.

from pyspark.sql.functions import sum as _sum, when, col, lit, count as _count

print("=" * 80)
print("6.4) COMPLETENESS CHECKS")
print("    Measuring missingness per column, per row, and critical field coverage")
print("=" * 80)

total_rows = df.count()
total_cols = len(df.columns)
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.4.1) Per-Column Missingness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.4.1) Per-Column Missingness")

col_null_data = []
for c in df.columns:
    null_cnt = df.filter(col(c).isNull()).count()
    pct = (null_cnt / total_rows * 100) if total_rows > 0 else 0
    icon = "✅" if null_cnt == 0 else ("⚠️ " if pct < 5 else "🚨")
    print(f"   {icon} {c}: {null_cnt:,} nulls ({pct:.2f}%)")
    col_null_data.append({"Column": c, "Null Count": null_cnt, "Null %": round(pct, 2), "Status": "PASS" if null_cnt == 0 else "FINDING"})

findings.extend(col_null_data)
col_null_df = pd.DataFrame(col_null_data)
print("\n")
display(col_null_df)

# ══════════════════════════════════════════════════════════════════════════════
# 6.4.2) Per-Row Missingness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.4.2) Per-Row Missingness")
print(f"   Rows with >50% null columns flagged as excessively incomplete")

# Count nulls per row
null_expr = sum([when(col(c).isNull(), lit(1)).otherwise(lit(0)) for c in df.columns])
df_with_null_count = df.withColumn("_null_count", null_expr)

# Rows with >50% columns null
threshold = total_cols * 0.5
excessive_null_rows = df_with_null_count.filter(col("_null_count") > threshold).count()

print(f"   Total columns: {total_cols}")
print(f"   Threshold: >{threshold:.0f} nulls per row (>50%)")
icon = "✅" if excessive_null_rows == 0 else "⚠️ "
status = "PASS" if excessive_null_rows == 0 else "FINDING"
print(f"   {icon} Rows with >50% null: {excessive_null_rows:,} / {total_rows:,} — {status}")

# Distribution of null counts per row
print("\n   Null-count distribution per row:")
null_dist = df_with_null_count.groupBy("_null_count").count().orderBy("_null_count")
display(null_dist)

# ══════════════════════════════════════════════════════════════════════════════
# 6.4.3) Critical Field Coverage
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.4.3) Critical Field Coverage")
print("   Required fields (per business rules) must be non-null")

critical_fields = ["user_id", "business_id", "text", "compliment_count", "tip_timestamp", "tip_date"]
critical_findings = []

for c in critical_fields:
    null_cnt = df.filter(col(c).isNull()).count()
    status = "PASS" if null_cnt == 0 else "CRITICAL FINDING"
    icon = "✅" if null_cnt == 0 else "🚨"
    print(f"   {icon} {c}: {null_cnt:,} nulls — {status}")
    critical_findings.append({"Critical Field": c, "Null Count": null_cnt, "Status": status})

print("\n")
display(pd.DataFrame(critical_findings))

# ══ Overall Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)

total_null_cells = sum(f["Null Count"] for f in col_null_data)
total_cells = total_rows * total_cols
null_rate = (total_null_cells / total_cells * 100) if total_cells > 0 else 0

print(f"   Overall null cell rate: {total_null_cells:,} / {total_cells:,} ({null_rate:.4f}%)")
print(f"   Rows with >50% null: {excessive_null_rows:,}")
print(f"   Critical field violations: {sum(1 for f in critical_findings if f['Status'] != 'PASS')}")

all_pass = all(f.get("Status") == "PASS" for f in col_null_data + critical_findings)
if all_pass and excessive_null_rows == 0:
    print("✅ Dataset is fully complete — zero nulls across all columns.")
else:
    print("⚠️  Completeness issues detected — review above.")
print("COMPLETENESS CHECKS COMPLETE")
print("=" * 80)

In [0]:
# 6.5) Uniqueness Checks
# ─────────────────────────────────────────────────────────────────────
# Ensure no duplicate data and each record is uniquely identifiable.

from pyspark.sql.functions import count as _count, countDistinct

print("=" * 80)
print("6.5) UNIQUENESS CHECKS")
print("    Checking row-level, key-level, composite key, and distribution uniqueness")
print("=" * 80)

total_rows = df.count()
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.5.1) Row-Level Uniqueness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.5.1) Row-Level Uniqueness (entire rows duplicated)")

distinct_rows = df.distinct().count()
duplicate_rows = total_rows - distinct_rows
dup_pct = (duplicate_rows / total_rows * 100) if total_rows > 0 else 0

icon = "✅" if duplicate_rows == 0 else "⚠️ "
status = "PASS" if duplicate_rows == 0 else "FINDING"
print(f"   {icon} Total rows: {total_rows:,}, Distinct rows: {distinct_rows:,}")
print(f"   {icon} Duplicate rows: {duplicate_rows:,} ({dup_pct:.2f}%) — {status}")
findings.append({"Check": "Row-Level Uniqueness", "Total": total_rows, "Distinct": distinct_rows, "Duplicates": duplicate_rows, "Status": status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.5.2) Key-Level Uniqueness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.5.2) Key-Level Uniqueness")
print("   (user_id, business_id) pair cardinality — tip table has no single-column primary key")

rid_distinct = df.select("user_id", "business_id").distinct().count()
rid_total = total_rows
rid_dups = rid_total - rid_distinct
icon = "✅" if rid_dups == 0 else "⚠️ "
status = "PASS" if rid_dups == 0 else "FINDING"
print(f"   {icon} (user_id, business_id): {rid_distinct:,} distinct values / {rid_total:,} rows")
print(f"   {icon} Duplicate (user_id, business_id) pairs: {rid_dups:,} — {status}")
findings.append({"Check": "(user_id, business_id) uniqueness", "Total": rid_total, "Distinct": rid_distinct, "Duplicates": rid_dups, "Status": status})

print("\n   business_id and user_id are NOT expected to be unique individually")
bid_distinct = df.select("business_id").distinct().count()
uid_distinct = df.select("user_id").distinct().count()
print(f"   business_id: {bid_distinct:,} distinct values / {rid_total:,} rows")
print(f"   user_id:     {uid_distinct:,} distinct values / {rid_total:,} rows")
findings.append({"Check": "business_id uniqueness", "Total": rid_total, "Distinct": bid_distinct, "Duplicates": rid_total - bid_distinct, "Status": "INFO (expected non-unique)"})
findings.append({"Check": "user_id uniqueness", "Total": rid_total, "Distinct": uid_distinct, "Duplicates": rid_total - uid_distinct, "Status": "INFO (expected non-unique)"})

# ══════════════════════════════════════════════════════════════════════════════
# 6.5.3) Composite Key Uniqueness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.5.3) Composite Key Uniqueness")
print("   (user_id, business_id, tip_timestamp, text) is a useful collision check")

composite_distinct = df.select("user_id", "business_id", "tip_timestamp", "text").distinct().count()
composite_dups = total_rows - composite_distinct

icon = "✅" if composite_dups == 0 else "⚠️ "
status = "PASS" if composite_dups == 0 else "FINDING"
print(f"   {icon} Composite key distinct: {composite_distinct:,} / {total_rows:,}")
print(f"   {icon} Composite key duplicates: {composite_dups:,} — {status}")

if composite_dups > 0:
    print("\n   Top duplicate composite keys:")
    dup_keys = (
        df.groupBy("user_id", "business_id", "tip_timestamp", "text")
        .agg(_count("*").alias("cnt"))
        .filter(col("cnt") > 1)
        .orderBy(col("cnt").desc())
        .limit(10)
    )
    display(dup_keys)

findings.append({"Check": "Composite Key (user + business + timestamp + text)", "Total": total_rows, "Distinct": composite_distinct, "Duplicates": composite_dups, "Status": status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.5.4) Distribution Uniqueness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.5.4) Distribution Uniqueness")
print("   Distinct value counts — flagging suspiciously low or high cardinality")

dist_data = []
for c in df.columns:
    distinct_cnt = df.select(c).distinct().count()
    cardinality_ratio = distinct_cnt / total_rows if total_rows > 0 else 0

    # Flag: single-value columns or unexpectedly high cardinality for categoricals
    if distinct_cnt == 1:
        flag = "SINGLE VALUE (constant)"
    elif distinct_cnt == total_rows:
        flag = "FULLY UNIQUE"
    elif cardinality_ratio > 0.95:
        flag = "NEAR-UNIQUE (>95%)"
    else:
        flag = "OK"

    dist_data.append({"Column": c, "Distinct Values": distinct_cnt, "Cardinality Ratio": round(cardinality_ratio, 4), "Flag": flag})

dist_df = pd.DataFrame(dist_data)
display(dist_df)

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f.get("Status") == "FINDING")
print(f"\n{len(findings)} uniqueness checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ No unexpected duplicates detected.")
else:
    print(f"⚠️  {violation_count} uniqueness issue(s) detected — review above.")
print("UNIQUENESS CHECKS COMPLETE")
print("=" * 80)

In [0]:
# 6.6) Cross-Column Consistency Checks
# ─────────────────────────────────────────────────────────────────────
# Validate logical relationships and contradictions across columns within a row.
# Every derived column must agree with its source timestamp.

from pyspark.sql.functions import (
    year, month, dayofmonth, dayofyear, dayofweek, hour, minute, second,
    quarter, weekofyear, date_format, last_day, col, when, lit, floor
)

print("=" * 80)
print("6.6) CROSS-COLUMN CONSISTENCY CHECKS")
print("    Validating derived columns match their source timestamp")
print("=" * 80)

total_rows = df.count()
ts_col = "tip_timestamp"
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.6.1) Logical Relationship Validation
#   Derived columns must match their source timestamp exactly.
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.6.1) Logical Relationship Validation")

# Define: (derived_col, expected_expression, description)
derivation_checks = [
    ("tip_date",         col(ts_col).cast("date"),                          "date portion of timestamp"),
    ("tip_year",         year(col(ts_col)),                                 "year(timestamp)"),
    ("tip_month",        month(col(ts_col)),                                "month(timestamp)"),
    ("tip_day",          dayofmonth(col(ts_col)),                           "dayofmonth(timestamp)"),
    ("tip_day_of_year",  dayofyear(col(ts_col)),                            "dayofyear(timestamp)"),
    ("tip_dow_number",   dayofweek(col(ts_col)),                            "dayofweek(timestamp)"),
    ("tip_quarter",      quarter(col(ts_col)),                              "quarter(timestamp)"),
    ("tip_week_of_year", weekofyear(col(ts_col)),                           "weekofyear(timestamp)"),
    ("tip_hour",         hour(col(ts_col)),                                 "hour(timestamp)"),
    ("tip_minute",       minute(col(ts_col)),                               "minute(timestamp)"),
    ("tip_second",       second(col(ts_col)),                               "second(timestamp)"),
    ("tip_month_name",   date_format(col(ts_col), "MMMM"),                  "date_format(ts, 'MMMM')"),
    ("tip_month_abbr",   date_format(col(ts_col), "MMM"),                   "date_format(ts, 'MMM')"),
    ("tip_dow_name",     date_format(col(ts_col), "EEEE"),                  "date_format(ts, 'EEEE')"),
    ("tip_dow_abbr",     date_format(col(ts_col), "EEE"),                   "date_format(ts, 'EEE')"),
    ("tip_year_month",   date_format(col(ts_col), "yyyy-MM"),               "date_format(ts, 'yyyy-MM')"),
]

for derived_col, expected_expr, desc in derivation_checks:
    mismatch_count = df.filter(col(derived_col) != expected_expr).count()
    status = "PASS" if mismatch_count == 0 else "FINDING"
    icon = "✅" if mismatch_count == 0 else "⚠️ "
    print(f"   {icon} {derived_col} == {desc}: {mismatch_count:,} mismatches — {status}")
    findings.append({"Check": f"{derived_col} == {desc}", "Mismatches": mismatch_count, "Status": status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.6.2) Derived-Column Logic Checks
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.6.2) Derived-Column Logic Checks")

logic_checks = [
    # half_year: month 1-6 → 1, month 7-12 → 2
    ("tip_half_year vs month",
     ((col("tip_month") <= 6) & (col("tip_half_year") != 1)) |
     ((col("tip_month") > 6) & (col("tip_half_year") != 2))),

    # half_year_name: month 1-6 → H1, month 7-12 → H2
    ("tip_half_year_name vs month",
     ((col("tip_month") <= 6) & (col("tip_half_year_name") != "H1")) |
     ((col("tip_month") > 6) & (col("tip_half_year_name") != "H2"))),

    # quarter_name matches quarter
    ("tip_quarter_name vs quarter",
     col("tip_quarter_name") != concat(lit("Q"), col("tip_quarter").cast("string"))),

    # AM/PM vs hour
    ("tip_am_pm vs hour",
     ((col("tip_hour") < 12) & (col("tip_am_pm") != "AM")) |
     ((col("tip_hour") >= 12) & (col("tip_am_pm") != "PM"))),

    # time_of_day vs hour
    ("tip_time_of_day vs hour",
     ~(
         (col("tip_hour").between(5, 11) & (col("tip_time_of_day") == "Morning")) |
         (col("tip_hour").between(12, 16) & (col("tip_time_of_day") == "Afternoon")) |
         (col("tip_hour").between(17, 20) & (col("tip_time_of_day") == "Evening")) |
         ((~col("tip_hour").between(5, 20)) & (col("tip_time_of_day") == "Night"))
     )),

    # hour_12 vs hour
    ("tip_hour_12 vs hour",
     ~(
         ((col("tip_hour") == 0) & (col("tip_hour_12") == 12)) |
         ((col("tip_hour").between(1, 12)) & (col("tip_hour_12") == col("tip_hour"))) |
         ((col("tip_hour") > 12) & (col("tip_hour_12") == (col("tip_hour") - 12)))
     )),

    # hour_bin_4h vs hour
    ("tip_hour_bin_4h vs hour",
     col("tip_hour_bin_4h") != (floor(col("tip_hour") / 4) * 4).cast("int")),

    # is_month_start vs day
    ("is_month_start vs day",
     col("is_month_start") != (col("tip_day") == 1)),

    # is_month_end vs date
    ("is_month_end vs date",
     col("is_month_end") != (col("tip_date") == last_day(col("tip_date")))),

    # is_year_start
    ("is_year_start vs month/day",
     col("is_year_start") != ((col("tip_month") == 1) & (col("tip_day") == 1))),

    # is_year_end
    ("is_year_end vs month/day",
     col("is_year_end") != ((col("tip_month") == 12) & (col("tip_day") == 31))),

    # is_weekend vs dow_number (1=Sun, 7=Sat)
    ("is_weekend vs dow_number",
     col("is_weekend") != col("tip_dow_number").isin(1, 7)),

    # is_weekday is complement of is_weekend
    ("is_weekday vs is_weekend",
     col("is_weekday") == col("is_weekend")),

    # month_start_date
    ("tip_month_start_date vs date",
     col("tip_month_start_date") != trunc(col("tip_date"), "month")),

    # month_end_date
    ("tip_month_end_date vs date",
     col("tip_month_end_date") != last_day(col("tip_date"))),
]

for desc, condition in logic_checks:
    mismatch_count = df.filter(condition).count()
    status = "PASS" if mismatch_count == 0 else "FINDING"
    icon = "✅" if mismatch_count == 0 else "⚠️ "
    print(f"   {icon} {desc}: {mismatch_count:,} mismatches — {status}")
    findings.append({"Check": desc, "Mismatches": mismatch_count, "Status": status})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} cross-column checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ All derived columns are perfectly consistent with their source timestamp.")
else:
    print(f"⚠️  {violation_count} cross-column inconsistency(ies) — review above.")
print("CROSS-COLUMN CONSISTENCY CHECKS COMPLETE")
print("=" * 80)

In [0]:
# 6.7) Referential Integrity Checks
# ─────────────────────────────────────────────────────────────────────
# Ensure relationships between tables are valid.
# business_id in tip should exist in yelp_business, and user_id in tip should exist in yelp_user.

print("=" * 80)
print("6.7) REFERENTIAL INTEGRITY CHECKS")
print("    Verifying foreign key relationships with parent tables")
print("=" * 80)

findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.7.1) FK Validity — business_id in yelp_business
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.7.1) FK Validity: tip.business_id → yelp_business.business_id")

try:
    df_business = spark.table("yelp_poc_uc.bronze_final.yelp_business")
    business_ids = df_business.select("business_id").distinct()

    # Distinct review business_ids
    tip_bids = df.select("business_id").distinct()
    total_tip_bids = tip_bids.count()
    total_business_bids = business_ids.count()

    print(f"   Distinct business_ids in tip:   {total_tip_bids:,}")
    print(f"   Distinct business_ids in business: {total_business_bids:,}")

    # Orphan records: review business_ids NOT in business table
    orphan_bids = tip_bids.join(business_ids, "business_id", "left_anti")
    orphan_count = orphan_bids.count()

    icon = "✅" if orphan_count == 0 else "⚠️ "
    status = "PASS" if orphan_count == 0 else "FINDING"
    print(f"\n   {icon} Orphan business_ids (in tip but NOT in business): {orphan_count:,} — {status}")

    if orphan_count > 0:
        orphan_samples = [r[0] for r in orphan_bids.limit(10).collect()]
        print(f"   Orphan examples: {orphan_samples}")
        # Count affected review rows
        orphan_rows = df.join(orphan_bids, "business_id", "inner").count()
        print(f"   Affected tip rows: {orphan_rows:,}")

    findings.append({"Check": "FK: tip.business_id → business", "Orphan Keys": orphan_count, "Status": status})

    # Missing parent records: business_ids in business but NOT in tip
    missing_reviews = business_ids.join(tip_bids, "business_id", "left_anti")
    missing_count = missing_reviews.count()
    print(f"\n   ℹ️  Businesses with ZERO tips (in business but not in tip): {missing_count:,}")
    print(f"      ({missing_count / total_business_bids * 100:.1f}% of businesses have no tip records)")
    findings.append({"Check": "Businesses with zero tips", "Orphan Keys": missing_count, "Status": "INFO"})

    # 6.7.2) FK Validity — user_id in yelp_user
    print(f"\n{'─'*60}")
    print("📌 6.7.2) FK Validity: tip.user_id → yelp_user.user_id")
    df_user = spark.table("yelp_poc_uc.bronze_final.yelp_user")
    user_ids = df_user.select("user_id").distinct()
    tip_uids = df.select("user_id").distinct()
    total_tip_uids = tip_uids.count()
    total_user_uids = user_ids.count()

    print(f"   Distinct user_ids in tip: {total_tip_uids:,}")
    print(f"   Distinct user_ids in user:   {total_user_uids:,}")

    orphan_uids = tip_uids.join(user_ids, "user_id", "left_anti")
    orphan_uid_count = orphan_uids.count()
    icon = "✅" if orphan_uid_count == 0 else "⚠️ "
    status = "PASS" if orphan_uid_count == 0 else "FINDING"
    print(f"\n   {icon} Orphan user_ids (in tip but NOT in user): {orphan_uid_count:,} — {status}")
    if orphan_uid_count > 0:
        orphan_samples = [r[0] for r in orphan_uids.limit(10).collect()]
        print(f"   Orphan examples: {orphan_samples}")
    findings.append({"Check": "FK: tip.user_id → user", "Orphan Keys": orphan_uid_count, "Status": status})

except Exception as e:
    print(f"   ⚠️  Could not load parent business/user tables: {e}")
    findings.append({"Check": "FK: tip.business_id → business", "Orphan Keys": "N/A", "Status": "SKIPPED"})

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f.get("Status") == "FINDING")
if violation_count == 0:
    print("✅ Referential integrity is intact — all business_ids and user_ids exist in their parent tables.")
else:
    print(f"⚠️  {violation_count} referential integrity issue(s) — review above.")
print("REFERENTIAL INTEGRITY CHECKS COMPLETE")
print("=" * 80)

In [0]:
# 6.8) Business Rule & Accuracy Checks
# ─────────────────────────────────────────────────────────────────────
# Domain-specific rules, statistical anomalies, and distribution checks.

from pyspark.sql.functions import count as _count, col, stddev, mean, percentile_approx
import numpy as np

print("=" * 80)
print("6.8) BUSINESS RULE & ACCURACY CHECKS")
print("    Domain-specific rules, statistical outliers, and distribution anomalies")
print("=" * 80)

total_rows = df.count()
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.8.1) Business Rule Violations
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.8.1) Business Rule Violations")

# Rule 1: Yelp tip year must be ≥ 2004 (Yelp founded) and ≤ current year
print(f"\n   Rule 1: tip_year must be in [2004, 2026]")
bad_year = df.filter((col("tip_year") < 2004) | (col("tip_year") > 2026)).count()
icon = "✅" if bad_year == 0 else "⚠️ "
status = "PASS" if bad_year == 0 else "FINDING"
print(f"   {icon} Violations: {bad_year:,} — {status}")
if bad_year > 0:
    bad_years = [r[0] for r in df.filter((col("tip_year") < 2004) | (col("tip_year") > 2026)).select("tip_year").distinct().collect()]
    print(f"   Bad years: {bad_years}")
findings.append({"Check": "Year in [2004, 2026]", "Violations": bad_year, "Status": status})

# Rule 2: No empty tip records
print(f"\n   Rule 2: No empty tip records (all rows have valid timestamp and text)")
empty_ts = df.filter(col("tip_timestamp").isNull() | col("text").isNull() | (trim(col("text")) == "")).count()
icon = "✅" if empty_ts == 0 else "⚠️ "
status = "PASS" if empty_ts == 0 else "FINDING"
print(f"   {icon} Null timestamps: {empty_ts:,} — {status}")
findings.append({"Check": "No null timestamps", "Violations": empty_ts, "Status": status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.8.2) Statistical Outlier Detection
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.8.2) Statistical Outlier Detection")
print("   Analyzing tip length distribution for outliers")

tip_metrics = df.withColumn("tip_text_length", length(col("text")))

stats = tip_metrics.agg(
    mean("tip_text_length").alias("avg"),
    stddev("tip_text_length").alias("std"),
    spark_min("tip_text_length").alias("min"),
    spark_max("tip_text_length").alias("max"),
    percentile_approx("tip_text_length", 0.25).alias("p25"),
    percentile_approx("tip_text_length", 0.50).alias("median"),
    percentile_approx("tip_text_length", 0.75).alias("p75"),
    percentile_approx("tip_text_length", 0.99).alias("p99"),
).collect()[0]

avg_val = stats["avg"]
std_val = stats["std"]
print(f"\n   Tip text length:")
print(f"      Mean:   {avg_val:.1f}")
print(f"      Std:    {std_val:.1f}")
print(f"      Min:    {stats['min']}")
print(f"      P25:    {stats['p25']}")
print(f"      Median: {stats['median']}")
print(f"      P75:    {stats['p75']}")
print(f"      P99:    {stats['p99']}")
print(f"      Max:    {stats['max']}")

# Z-score outliers: > 3 standard deviations from mean
z_threshold = avg_val + 3 * std_val
outlier_tips = tip_metrics.filter(col("tip_text_length") > z_threshold)
outlier_count = outlier_tips.count()

print(f"\n   Z-score threshold (mean + 3σ): {z_threshold:.0f}")
print(f"   Tips with text_length > {z_threshold:.0f}: {outlier_count:,}")

if outlier_count > 0:
    print(f"\n   Top 10 longest outlier tips:")
    display(outlier_tips.select("user_id", "business_id", "compliment_count", "tip_text_length").orderBy(col("tip_text_length").desc()).limit(10))

findings.append({"Check": "Tip-length outliers (Z>3)", "Violations": outlier_count, "Status": "INFO" if outlier_count > 0 else "PASS"})

# IQR-based outliers
iqr = stats["p75"] - stats["p25"]
upper_fence = stats["p75"] + 1.5 * iqr
iqr_outliers = tip_metrics.filter(col("tip_text_length") > upper_fence).count()
print(f"\n   IQR upper fence (P75 + 1.5*IQR): {upper_fence:.0f}")
print(f"   Tips above IQR fence: {iqr_outliers:,}")
findings.append({"Check": "Tip-length outliers (IQR)", "Violations": iqr_outliers, "Status": "INFO" if iqr_outliers > 0 else "PASS"})

# ══════════════════════════════════════════════════════════════════════════════
# 6.8.3) Distribution Anomalies
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.8.3) Distribution Anomalies")

# Year distribution: check for sudden spikes/drops
print("\n   Year distribution (tip volume per year):")
year_dist = df.groupBy("tip_year").agg(_count("*").alias("cnt")).orderBy("tip_year")
display(year_dist)

# Day-of-week distribution: check for imbalance
print("\n   Day-of-week distribution:")
dow_dist = df.groupBy("tip_dow_name").agg(_count("*").alias("cnt")).orderBy(col("cnt").desc())
display(dow_dist)

# Hour distribution: check for anomalies
print("\n   Hour distribution:")
hour_dist = df.groupBy("tip_hour").agg(_count("*").alias("cnt")).orderBy("tip_hour")
display(hour_dist)

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} business rule / accuracy checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ No business rule violations detected. Statistical distributions shown above for tip data.")
else:
    print(f"⚠️  {violation_count} business rule violation(s) — review above.")
print("BUSINESS RULE & ACCURACY CHECKS COMPLETE")
print("=" * 80)

In [0]:
# 6.9) Timeliness Checks
# ─────────────────────────────────────────────────────────────────────
# Monitor data freshness, latency, and temporal logic.

from pyspark.sql.functions import (
    min as spark_min, max as spark_max, datediff, current_date,
    current_timestamp, count as _count, col, year, month, lit
)
from pyspark.sql import Window

print("=" * 80)
print("6.9) TIMELINESS CHECKS")
print("    Data freshness, temporal consistency, and temporal gaps")
print("=" * 80)

total_rows = df.count()
findings = []

# ══════════════════════════════════════════════════════════════════════════════
# 6.9.1) Data Freshness
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.9.1) Data Freshness")

ts_range = df.agg(
    spark_min("tip_timestamp").alias("earliest"),
    spark_max("tip_timestamp").alias("latest"),
).collect()[0]

earliest = ts_range["earliest"]
latest = ts_range["latest"]

# Days since latest review
days_since = df.agg(
    datediff(current_date(), spark_max(col("tip_date"))).alias("days_stale")
).collect()[0]["days_stale"]

print(f"   Earliest tip: {earliest}")
print(f"   Latest tip:   {latest}")
print(f"   Days since latest tip: {days_since:,} days")
print(f"   Data spans: {earliest} → {latest}")

# Freshness assessment
if days_since <= 30:
    print(f"   ✅ Data is fresh (within 30 days)")
    freshness_status = "PASS"
elif days_since <= 365:
    print(f"   ⚠️  Data is moderately stale ({days_since} days old)")
    freshness_status = "INFO"
else:
    print(f"   ⚠️  Data is significantly stale ({days_since} days / {days_since//365} years old)")
    freshness_status = "INFO"

findings.append({"Check": "Data Freshness", "Detail": f"Latest: {latest}, {days_since} days ago", "Status": freshness_status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.9.2) Temporal Consistency
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.9.2) Temporal Consistency")

# Future-dated tips
future_checkins = df.filter(col("tip_timestamp") > current_timestamp()).count()
icon = "✅" if future_checkins == 0 else "⚠️ "
status = "PASS" if future_checkins == 0 else "FINDING"
print(f"   {icon} Future-dated tips: {future_checkins:,} — {status}")
findings.append({"Check": "Future-dated tips", "Detail": f"{future_checkins:,} future timestamps", "Status": status})

# Date columns consistency: month_start_date <= tip_date <= month_end_date
date_order_bad = df.filter(
    (col("tip_date") < col("tip_month_start_date")) |
    (col("tip_date") > col("tip_month_end_date"))
).count()
icon = "✅" if date_order_bad == 0 else "⚠️ "
status = "PASS" if date_order_bad == 0 else "FINDING"
print(f"   {icon} Date outside its own month range: {date_order_bad:,} — {status}")
findings.append({"Check": "Date within month range", "Detail": f"{date_order_bad:,} violations", "Status": status})

# year_start_date should be Jan 1 of the same year
year_start_bad = df.filter(
    col("tip_year_start_date") != trunc(col("tip_date"), "year")
).count()
icon = "✅" if year_start_bad == 0 else "⚠️ "
status = "PASS" if year_start_bad == 0 else "FINDING"
print(f"   {icon} year_start_date != trunc(date, 'year'): {year_start_bad:,} — {status}")
findings.append({"Check": "year_start_date consistency", "Detail": f"{year_start_bad:,} violations", "Status": status})

# ══════════════════════════════════════════════════════════════════════════════
# 6.9.3) Temporal Gaps
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*60}")
print("📌 6.9.3) Temporal Gaps")
print("   Checking for missing year-month periods in the data")

# Get actual year-months present
actual_ym = set(
    r[0] for r in df.select("tip_year_month").distinct().collect()
)

# Generate expected year-months from min to max
min_ym = min(actual_ym)
max_ym = max(actual_ym)

min_year, min_month = int(min_ym[:4]), int(min_ym[5:])
max_year, max_month = int(max_ym[:4]), int(max_ym[5:])

expected_ym = set()
y, m = min_year, min_month
while (y, m) <= (max_year, max_month):
    expected_ym.add(f"{y:04d}-{m:02d}")
    m += 1
    if m > 12:
        m = 1
        y += 1

missing_ym = sorted(expected_ym - actual_ym)

print(f"   Data range: {min_ym} → {max_ym}")
print(f"   Expected year-months: {len(expected_ym)}")
print(f"   Actual year-months:   {len(actual_ym)}")
print(f"   Missing year-months:  {len(missing_ym)}")

if missing_ym:
    print(f"\n   ⚠️  Missing periods:")
    # Group consecutive missing months for readability
    for ym in missing_ym:
        print(f"      • {ym}")
    findings.append({"Check": "Temporal Gaps (year-month)", "Detail": f"{len(missing_ym)} missing periods", "Status": "FINDING"})
else:
    print(f"   ✅ No temporal gaps — all year-months are present")
    findings.append({"Check": "Temporal Gaps (year-month)", "Detail": "No gaps", "Status": "PASS"})

# Year-level gaps
actual_years = sorted(set(int(ym[:4]) for ym in actual_ym))
expected_years = list(range(min(actual_years), max(actual_years) + 1))
missing_years = [y for y in expected_years if y not in actual_years]

if missing_years:
    print(f"\n   ⚠️  Missing entire years: {missing_years}")
    findings.append({"Check": "Temporal Gaps (year)", "Detail": f"Missing years: {missing_years}", "Status": "FINDING"})
else:
    print(f"\n   ✅ All years from {min(actual_years)} to {max(actual_years)} are represented")
    findings.append({"Check": "Temporal Gaps (year)", "Detail": "No gaps", "Status": "PASS"})

# Volume per year for trend review
print("\n   Volume trend by year:")
year_volume = df.groupBy("tip_year").agg(_count("*").alias("cnt")).orderBy("tip_year")
display(year_volume)

# ══ Summary ══
print(f"\n{'='*80}")
print("SUMMARY")
print("=" * 80)
findings_df = pd.DataFrame(findings)
display(findings_df)

violation_count = sum(1 for f in findings if f["Status"] == "FINDING")
print(f"\n{len(findings)} timeliness checks performed, {violation_count} finding(s)")
if violation_count == 0:
    print("✅ Data is temporally consistent with no gaps.")
else:
    print(f"⚠️  {violation_count} timeliness issue(s) — review above.")
print("TIMELINESS CHECKS COMPLETE")
print("=" * 80)